# Step 23 — Real Test Data Validation

## Objective

Validate the compatibility between the historical training pipeline and the
actual Rossmann `test.csv` dataset.

This analysis investigates:

- Raw dataset schemas
- Train/test column differences
- Store metadata availability
- Model feature compatibility
- Availability of historical Sales for lag features
- Rolling feature feasibility
- Real test prediction requirements
- Submission format compatibility

The goal is to ensure that the production model can be safely applied to
the actual unseen test dataset without data leakage or schema mismatch.

## 23A — Raw Data Schema Audit

We first inspect the four original datasets:

- `train.csv`
- `test.csv`
- `store.csv`
- `sample_submission.csv`

The raw datasets are treated as the source of truth for the real inference
pipeline.

In [4]:
import pandas as pd

train = pd.read_csv("../data/raw/train.csv")
test = pd.read_csv("../data/raw/test.csv")
store = pd.read_csv("../data/raw/store.csv")
submission = pd.read_csv("../data/raw/sample_submission.csv")

print("Train:", train.shape)
print("Test:", test.shape)
print("Store:", store.shape)
print("Submission:", submission.shape)

Train: (1017209, 9)
Test: (41088, 8)
Store: (1115, 10)
Submission: (41088, 2)


C:\Users\HP\AppData\Local\Temp\ipykernel_18044\185331390.py:3: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv("../data/raw/train.csv")


### 23A Observation

The raw datasets have different purposes and therefore different schemas.

In particular:

- `train.csv` contains the target `Sales` and historical `Customers`.
- `test.csv` does not contain `Sales` or `Customers`.
- `store.csv` contains store-level characteristics required to enrich both
  training and test data.
- `sample_submission.csv` defines the required final prediction format.

Therefore, the existing training/evaluation pipeline cannot simply be applied
to the raw test dataset without validating feature availability first.

##### 23B — Schema comparison

In [5]:
print("Train columns:")
print(train.columns.tolist())

print("\nTest columns:")
print(test.columns.tolist())

print("\nStore columns:")
print(store.columns.tolist())

print("\nSubmission columns:")
print(submission.columns.tolist())

Train columns:
['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday']

Test columns:
['Id', 'Store', 'DayOfWeek', 'Date', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday']

Store columns:
['Store', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval']

Submission columns:
['Id', 'Sales']


##### 23C — Find train-only and test-only columns

In [6]:
train_only = sorted(set(train.columns) - set(test.columns))
test_only = sorted(set(test.columns) - set(train.columns))

print("Train-only columns:")
print(train_only)

print("\nTest-only columns:")
print(test_only)

Train-only columns:
['Customers', 'Sales']

Test-only columns:
['Id']


##### 23D — Check whether Customers entered our model

In [8]:
train_split = pd.read_csv("../data/splits/train.csv")
validation = pd.read_csv("../data/splits/validation.csv")
test_split = pd.read_csv("../data/splits/test.csv")

print("Training columns:")
print(train_split.columns.tolist())

print("\nNumber of training columns:", len(train_split.columns))

Training columns:
['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval', 'Year', 'Month', 'Day', 'WeekOfYear', 'HasCompetitionDistance', 'HasPromo2', 'StoreAge', 'Sales_Lag_1', 'Sales_Lag_7', 'Sales_Lag_14', 'Sales_Rolling_Mean_7', 'Sales_Rolling_Mean_14', 'Sales_Rolling_Mean_30']

Number of training columns: 31


##### 23E — Compare current model features with raw Kaggle test

In [13]:
feature_columns = [
    "Store",
    "DayOfWeek",
    "Open",
    "Promo",
    "SchoolHoliday",
    "CompetitionDistance",
    "StoreType",
    "Assortment",
    "Year",
    "Month",
    "Day",
    "WeekOfYear",
    "HasCompetitionDistance",
    "HasPromo2",
    "StoreAge",
    "Sales_Lag_1",
    "Sales_Lag_7",
    "Sales_Lag_14",
    "Sales_Rolling_Mean_7",
    "Sales_Rolling_Mean_14",
    "Sales_Rolling_Mean_30",
]

print("Number of model features:", len(feature_columns))
print("\nModel features:")
for i, col in enumerate(feature_columns, 1):
    print(f"{i:2}. {col}")

Number of model features: 21

Model features:
 1. Store
 2. DayOfWeek
 3. Open
 4. Promo
 5. SchoolHoliday
 6. CompetitionDistance
 7. StoreType
 8. Assortment
 9. Year
10. Month
11. Day
12. WeekOfYear
13. HasCompetitionDistance
14. HasPromo2
15. StoreAge
16. Sales_Lag_1
17. Sales_Lag_7
18. Sales_Lag_14
19. Sales_Rolling_Mean_7
20. Sales_Rolling_Mean_14
21. Sales_Rolling_Mean_30


##### 23F — Compare with raw test

In [14]:
raw_test_columns = set(test.columns)

directly_available = [
    col for col in feature_columns
    if col in raw_test_columns
]

not_directly_available = [
    col for col in feature_columns
    if col not in raw_test_columns
]

print("Directly available in raw test:")
print(directly_available)

print("\nNot directly available in raw test:")
print(not_directly_available)

Directly available in raw test:
['Store', 'DayOfWeek', 'Open', 'Promo', 'SchoolHoliday']

Not directly available in raw test:
['CompetitionDistance', 'StoreType', 'Assortment', 'Year', 'Month', 'Day', 'WeekOfYear', 'HasCompetitionDistance', 'HasPromo2', 'StoreAge', 'Sales_Lag_1', 'Sales_Lag_7', 'Sales_Lag_14', 'Sales_Rolling_Mean_7', 'Sales_Rolling_Mean_14', 'Sales_Rolling_Mean_30']


##### 23G — Classify every feature

In [15]:
direct_features = [
    "Store",
    "DayOfWeek",
    "Open",
    "Promo",
    "SchoolHoliday",
]

store_features = [
    "CompetitionDistance",
    "StoreType",
    "Assortment",
    "HasCompetitionDistance",
    "HasPromo2",
    "StoreAge",
]

date_features = [
    "Year",
    "Month",
    "Day",
    "WeekOfYear",
]

lag_features = [
    "Sales_Lag_1",
    "Sales_Lag_7",
    "Sales_Lag_14",
    "Sales_Rolling_Mean_7",
    "Sales_Rolling_Mean_14",
    "Sales_Rolling_Mean_30",
]

print("Direct test features:", len(direct_features))
print("Store-derived features:", len(store_features))
print("Date-derived features:", len(date_features))
print("Historical-sales-derived features:", len(lag_features))

print("\nTotal:",
      len(direct_features)
      + len(store_features)
      + len(date_features)
      + len(lag_features))

Direct test features: 5
Store-derived features: 6
Date-derived features: 4
Historical-sales-derived features: 6

Total: 21


##### 23H — Inspect actual raw test dates

In [17]:
print("Raw test date range:")
print(test["Date"].min())
print(test["Date"].max())

print("\nTest records:", len(test))

print("\nFirst 10 test dates:")
print(test[["Id", "Store", "Date"]].head(10))

Raw test date range:
2015-08-01
2015-09-17

Test records: 41088

First 10 test dates:
   Id  Store        Date
0   1      1  2015-09-17
1   2      3  2015-09-17
2   3      7  2015-09-17
3   4      8  2015-09-17
4   5      9  2015-09-17
5   6     10  2015-09-17
6   7     11  2015-09-17
7   8     12  2015-09-17
8   9     13  2015-09-17
9  10     14  2015-09-17


In [18]:
print("\nLast 10 test dates:")
print(test[["Id", "Store", "Date"]].tail(10))


Last 10 test dates:
          Id  Store        Date
41078  41079   1104  2015-08-01
41079  41080   1105  2015-08-01
41080  41081   1106  2015-08-01
41081  41082   1107  2015-08-01
41082  41083   1109  2015-08-01
41083  41084   1111  2015-08-01
41084  41085   1112  2015-08-01
41085  41086   1113  2015-08-01
41086  41087   1114  2015-08-01
41087  41088   1115  2015-08-01


##### Step 23I — Determine whether historical Sales can support the real test

##### Step 23I.1 — First verify the boundary

In [19]:
train_raw = pd.read_csv("../data/raw/train.csv", low_memory=False)
test_raw = pd.read_csv("../data/raw/test.csv", low_memory=False)

train_raw["Date"] = pd.to_datetime(train_raw["Date"])
test_raw["Date"] = pd.to_datetime(test_raw["Date"])

print("TRAIN:")
print(train_raw["Date"].min(), "→", train_raw["Date"].max())

print("\nTEST:")
print(test_raw["Date"].min(), "→", test_raw["Date"].max())

print("\nLast 10 training dates:")
print(
    train_raw[["Store", "Date", "Sales"]]
    .sort_values(["Store", "Date"])
    .tail(10)
)

TRAIN:
2013-01-01 00:00:00 → 2015-07-31 00:00:00

TEST:
2015-08-01 00:00:00 → 2015-09-17 00:00:00

Last 10 training dates:
       Store       Date  Sales
11149   1115 2015-07-22   5342
10034   1115 2015-07-23   6150
8919    1115 2015-07-24   5816
7804    1115 2015-07-25   6897
6689    1115 2015-07-26      0
5574    1115 2015-07-27  10712
4459    1115 2015-07-28   8093
3344    1115 2015-07-29   7661
2229    1115 2015-07-30   8405
1114    1115 2015-07-31   8680


In [20]:
print(
    "\nTraining records on 2015-07-31:",
    (train_raw["Date"] == "2015-07-31").sum()
)

print(
    "Test records on 2015-08-01:",
    (test_raw["Date"] == "2015-08-01").sum()
)


Training records on 2015-07-31: 1115
Test records on 2015-08-01: 856


##### Step 23I.2 — Check whether every test Store has historical Sales

In [21]:
train_stores = set(train_raw["Store"].unique())
test_stores = set(test_raw["Store"].unique())

print("Stores in train:", len(train_stores))
print("Stores in test:", len(test_stores))

print(
    "Test stores missing from train:",
    sorted(test_stores - train_stores)
)

Stores in train: 1115
Stores in test: 856
Test stores missing from train: []


In [22]:
last_train = (
    train_raw[train_raw["Date"] == train_raw["Date"].max()]
    ["Store"]
    .nunique()
)

test_store_count = test_raw["Store"].nunique()

print("Stores with data on final training date:", last_train)
print("Stores in test:", test_store_count)

Stores with data on final training date: 1115
Stores in test: 856


##### Step 23I.3 — Check actual historical lag availability

In [23]:
first_test_date = test_raw["Date"].min()

required_dates = [
    first_test_date - pd.Timedelta(days=1),
    first_test_date - pd.Timedelta(days=7),
    first_test_date - pd.Timedelta(days=14),
    first_test_date - pd.Timedelta(days=30),
]

print("First test date:", first_test_date)

for d in required_dates:
    count = (train_raw["Date"] == d).sum()
    print(f"{d.date()}: {count} training records")

First test date: 2015-08-01 00:00:00
2015-07-31: 1115 training records
2015-07-25: 1115 training records
2015-07-18: 1115 training records
2015-07-02: 1115 training records


##### Step 23J — Simulate Lag Availability Across Real Test

##### 23J.1 Check which lag dates fall inside train vs test

In [24]:
first_test_date = test_raw["Date"].min()
last_test_date = test_raw["Date"].max()

test_dates = pd.date_range(
    start=first_test_date,
    end=last_test_date,
    freq="D"
)

lag_days = [1, 7, 14]

for lag in lag_days:
    required_dates = test_dates - pd.Timedelta(days=lag)

    from_train = required_dates <= train_raw["Date"].max()
    from_test = required_dates > train_raw["Date"].max()

    print(f"\nSales_Lag_{lag}")
    print("Required dates from training:", from_train.sum())
    print("Required dates from test:", from_test.sum())


Sales_Lag_1
Required dates from training: 1
Required dates from test: 47

Sales_Lag_7
Required dates from training: 7
Required dates from test: 41

Sales_Lag_14
Required dates from training: 14
Required dates from test: 34


##### 23J.2 Show the transition explicitly

In [25]:
boundary = train_raw["Date"].max()

check_dates = pd.DataFrame({
    "TestDate": test_dates
})

check_dates["Lag_1_Date"] = (
    check_dates["TestDate"] - pd.Timedelta(days=1)
)

check_dates["Lag_7_Date"] = (
    check_dates["TestDate"] - pd.Timedelta(days=7)
)

check_dates["Lag_14_Date"] = (
    check_dates["TestDate"] - pd.Timedelta(days=14)
)

check_dates["Lag_1_Source"] = check_dates["Lag_1_Date"].apply(
    lambda x: "TRAIN" if x <= boundary else "UNKNOWN_TEST"
)

check_dates["Lag_7_Source"] = check_dates["Lag_7_Date"].apply(
    lambda x: "TRAIN" if x <= boundary else "UNKNOWN_TEST"
)

check_dates["Lag_14_Source"] = check_dates["Lag_14_Date"].apply(
    lambda x: "TRAIN" if x <= boundary else "UNKNOWN_TEST"
)

print(check_dates.head(20))

     TestDate Lag_1_Date Lag_7_Date Lag_14_Date  Lag_1_Source  Lag_7_Source  \
0  2015-08-01 2015-07-31 2015-07-25  2015-07-18         TRAIN         TRAIN   
1  2015-08-02 2015-08-01 2015-07-26  2015-07-19  UNKNOWN_TEST         TRAIN   
2  2015-08-03 2015-08-02 2015-07-27  2015-07-20  UNKNOWN_TEST         TRAIN   
3  2015-08-04 2015-08-03 2015-07-28  2015-07-21  UNKNOWN_TEST         TRAIN   
4  2015-08-05 2015-08-04 2015-07-29  2015-07-22  UNKNOWN_TEST         TRAIN   
5  2015-08-06 2015-08-05 2015-07-30  2015-07-23  UNKNOWN_TEST         TRAIN   
6  2015-08-07 2015-08-06 2015-07-31  2015-07-24  UNKNOWN_TEST         TRAIN   
7  2015-08-08 2015-08-07 2015-08-01  2015-07-25  UNKNOWN_TEST  UNKNOWN_TEST   
8  2015-08-09 2015-08-08 2015-08-02  2015-07-26  UNKNOWN_TEST  UNKNOWN_TEST   
9  2015-08-10 2015-08-09 2015-08-03  2015-07-27  UNKNOWN_TEST  UNKNOWN_TEST   
10 2015-08-11 2015-08-10 2015-08-04  2015-07-28  UNKNOWN_TEST  UNKNOWN_TEST   
11 2015-08-12 2015-08-11 2015-08-05  2015-07-29  UNK

In [26]:
daily_test_counts = (
    test_raw.groupby("Date")["Store"]
    .nunique()
)

print(daily_test_counts)

print("\nUnique daily counts:")
print(daily_test_counts.value_counts().sort_index())

Date
2015-08-01    856
2015-08-02    856
2015-08-03    856
2015-08-04    856
2015-08-05    856
2015-08-06    856
2015-08-07    856
2015-08-08    856
2015-08-09    856
2015-08-10    856
2015-08-11    856
2015-08-12    856
2015-08-13    856
2015-08-14    856
2015-08-15    856
2015-08-16    856
2015-08-17    856
2015-08-18    856
2015-08-19    856
2015-08-20    856
2015-08-21    856
2015-08-22    856
2015-08-23    856
2015-08-24    856
2015-08-25    856
2015-08-26    856
2015-08-27    856
2015-08-28    856
2015-08-29    856
2015-08-30    856
2015-08-31    856
2015-09-01    856
2015-09-02    856
2015-09-03    856
2015-09-04    856
2015-09-05    856
2015-09-06    856
2015-09-07    856
2015-09-08    856
2015-09-09    856
2015-09-10    856
2015-09-11    856
2015-09-12    856
2015-09-13    856
2015-09-14    856
2015-09-15    856
2015-09-16    856
2015-09-17    856
Name: Store, dtype: int64

Unique daily counts:
Store
856    48
Name: count, dtype: int64


In [27]:
store_test_counts = (
    test_raw.groupby("Store")["Date"]
    .count()
)

print("\nTest days per store:")
print(store_test_counts.describe())

print("\nStores by number of test days:")
print(store_test_counts.value_counts().sort_index())


Test days per store:
count    856.0
mean      48.0
std        0.0
min       48.0
25%       48.0
50%       48.0
75%       48.0
max       48.0
Name: Date, dtype: float64

Stores by number of test days:
Date
48    856
Name: count, dtype: int64


##### Step 23L — Validate Store Metadata

In [28]:
store_raw = pd.read_csv("../data/raw/store.csv", low_memory=False)

test_stores = set(test_raw["Store"].unique())
store_master_stores = set(store_raw["Store"].unique())

missing_store_metadata = test_stores - store_master_stores

print("Test stores:", len(test_stores))
print("Stores in store.csv:", len(store_master_stores))
print("Test stores missing from store.csv:", len(missing_store_metadata))

if missing_store_metadata:
    print(sorted(missing_store_metadata))
else:
    print("✓ All test stores have store metadata")

Test stores: 856
Stores in store.csv: 1115
Test stores missing from store.csv: 0
✓ All test stores have store metadata


In [29]:
duplicate_store_ids = store_raw["Store"].duplicated().sum()

print("Duplicate Store IDs in store.csv:", duplicate_store_ids)

Duplicate Store IDs in store.csv: 0


##### Step 23M — Build the first real test-day feature matrix

##### 23M.1 Load the data

In [35]:
import pandas as pd
import numpy as np

train_raw = pd.read_csv(
    "../data/raw/train.csv",
    low_memory=False
)

test_raw = pd.read_csv(
    "../data/raw/test.csv",
    low_memory=False
)

store_raw = pd.read_csv(
    "../data/raw/store.csv",
    low_memory=False
)

train_raw["Date"] = pd.to_datetime(train_raw["Date"])
test_raw["Date"] = pd.to_datetime(test_raw["Date"])

print("Train:", train_raw.shape)
print("Test:", test_raw.shape)
print("Store:", store_raw.shape)

Train: (1017209, 9)
Test: (41088, 8)
Store: (1115, 10)


##### 23M.2 Select the first test day

In [36]:
first_test_date = test_raw["Date"].min()

test_day = test_raw[
    test_raw["Date"] == first_test_date
].copy()

print("Test date:", first_test_date)
print("Rows:", len(test_day))
print("Unique stores:", test_day["Store"].nunique())

Test date: 2015-08-01 00:00:00
Rows: 856
Unique stores: 856


##### Step 23M.3 Add store information

In [37]:
test_day = test_day.merge(
    store_raw,
    on="Store",
    how="left",
    validate="many_to_one"
)

print("Shape after store merge:", test_day.shape)

print("\nMissing values in store metadata:")

metadata_columns = [
    "StoreType",
    "Assortment",
    "CompetitionDistance",
    "CompetitionOpenSinceMonth",
    "CompetitionOpenSinceYear",
    "Promo2",
    "Promo2SinceWeek",
    "Promo2SinceYear",
    "PromoInterval"
]

print(test_day[metadata_columns].isna().sum())

Shape after store merge: (856, 17)

Missing values in store metadata:
StoreType                      0
Assortment                     0
CompetitionDistance            2
CompetitionOpenSinceMonth    317
CompetitionOpenSinceYear     317
Promo2                         0
Promo2SinceWeek              359
Promo2SinceYear              359
PromoInterval                359
dtype: int64


##### 23M.4 Create date features

In [38]:
test_day["Year"] = test_day["Date"].dt.year
test_day["Month"] = test_day["Date"].dt.month
test_day["Day"] = test_day["Date"].dt.day
test_day["WeekOfYear"] = (
    test_day["Date"]
    .dt.isocalendar()
    .week
    .astype(int)
)

In [39]:
print(
    test_day[
        [
            "Date",
            "Year",
            "Month",
            "Day",
            "WeekOfYear"
        ]
    ].head()
)

        Date  Year  Month  Day  WeekOfYear
0 2015-08-01  2015      8    1          31
1 2015-08-01  2015      8    1          31
2 2015-08-01  2015      8    1          31
3 2015-08-01  2015      8    1          31
4 2015-08-01  2015      8    1          31


##### Step 23M.5 Create business features

In [40]:
test_day["HasCompetitionDistance"] = (
    test_day["CompetitionDistance"]
    .notna()
    .astype(int)
)

test_day["HasPromo2"] = (
    test_day["Promo2"]
    .fillna(0)
    .astype(int)
)

test_day["StoreAge"] = (
    test_day["Year"]
    - test_day["CompetitionOpenSinceYear"]
)

test_day["StoreAge"] = (
    test_day["StoreAge"]
    .clip(lower=0)
)

In [41]:
print(
    test_day[
        [
            "Store",
            "CompetitionDistance",
            "HasCompetitionDistance",
            "Promo2",
            "HasPromo2",
            "CompetitionOpenSinceYear",
            "StoreAge"
        ]
    ].head(10)
)

   Store  CompetitionDistance  HasCompetitionDistance  Promo2  HasPromo2  \
0      1               1270.0                       1       0          0   
1      3              14130.0                       1       1          1   
2      7              24000.0                       1       0          0   
3      8               7520.0                       1       0          0   
4      9               2030.0                       1       0          0   
5     10               3160.0                       1       0          0   
6     11                960.0                       1       1          1   
7     12               1070.0                       1       1          1   
8     13                310.0                       1       1          1   
9     14               1300.0                       1       1          1   

   CompetitionOpenSinceYear  StoreAge  
0                    2008.0       7.0  
1                    2006.0       9.0  
2                    2013.0       2.0  
3  

##### Step 23M.6 — Build the historical Sales lookup

In [48]:
# Historical sales sorted by Store and Date

sales_history = (
    train_raw[
        ["Store", "Date", "Sales"]
    ]
    .sort_values(["Store", "Date"])
    .copy()
)

print("Historical sales shape:", sales_history.shape)

print("\nHistorical date range:")
print(sales_history["Date"].min())
print(sales_history["Date"].max())

print("\nSales records on final training date:")
print(
    (sales_history["Date"] == "2015-07-31").sum()
)

Historical sales shape: (1017209, 3)

Historical date range:
2013-01-01 00:00:00
2015-07-31 00:00:00

Sales records on final training date:
1115


##### Step 23M.7 — Verify the three lag dates

In [49]:
lag_dates = {
    "Sales_Lag_1": pd.Timestamp("2015-07-31"),
    "Sales_Lag_7": pd.Timestamp("2015-07-25"),
    "Sales_Lag_14": pd.Timestamp("2015-07-18"),
}

for feature, date in lag_dates.items():

    count = (
        sales_history["Date"] == date
    ).sum()

    stores = (
        sales_history.loc[
            sales_history["Date"] == date,
            "Store"
        ].nunique()
    )

    print(
        f"{feature}: {date.date()} | "
        f"rows={count} | stores={stores}"
    )

Sales_Lag_1: 2015-07-31 | rows=1115 | stores=1115
Sales_Lag_7: 2015-07-25 | rows=1115 | stores=1115
Sales_Lag_14: 2015-07-18 | rows=1115 | stores=1115


##### Step 23M.8 — Calculate Aug 1 lag features

In [50]:
lag_features = test_day[
    ["Store"]
].copy()

for feature, date in lag_dates.items():

    lookup = (
        sales_history[
            sales_history["Date"] == date
        ][
            ["Store", "Sales"]
        ]
        .rename(
            columns={"Sales": feature}
        )
    )

    lag_features = lag_features.merge(
        lookup,
        on="Store",
        how="left",
        validate="one_to_one"
    )

print(lag_features.head())

   Store  Sales_Lag_1  Sales_Lag_7  Sales_Lag_14
0      1         5263         4364          4406
1      3         8314         3878          4258
2      7        15344         6574          6861
3      8         8492         3288          3813
4      9         8565         5330          5987


In [51]:
print("\nMissing lag values:")
print(
    lag_features[
        [
            "Sales_Lag_1",
            "Sales_Lag_7",
            "Sales_Lag_14"
        ]
    ].isna().sum()
)


Missing lag values:
Sales_Lag_1     0
Sales_Lag_7     0
Sales_Lag_14    0
dtype: int64


##### Step 23M.9 — Build the rolling features correctly

In [52]:
rolling_features = (
    sales_history[
        sales_history["Date"] <= "2015-07-31"
    ]
    .sort_values(["Store", "Date"])
    .copy()
)

rolling_features["Sales_Rolling_Mean_7"] = (
    rolling_features
    .groupby("Store")["Sales"]
    .transform(
        lambda x: x.shift(1).rolling(7).mean()
    )
)

rolling_features["Sales_Rolling_Mean_14"] = (
    rolling_features
    .groupby("Store")["Sales"]
    .transform(
        lambda x: x.shift(1).rolling(14).mean()
    )
)

rolling_features["Sales_Rolling_Mean_30"] = (
    rolling_features
    .groupby("Store")["Sales"]
    .transform(
        lambda x: x.shift(1).rolling(30).mean()
    )
)

In [53]:
rolling_windows = {}

for window, feature in [
    (7, "Sales_Rolling_Mean_7"),
    (14, "Sales_Rolling_Mean_14"),
    (30, "Sales_Rolling_Mean_30"),
]:

    cutoff = pd.Timestamp("2015-07-31")

    start_date = (
        cutoff - pd.Timedelta(days=window - 1)
    )

    subset = sales_history[
        (sales_history["Date"] >= start_date)
        & (sales_history["Date"] <= cutoff)
    ]

    rolling_windows[feature] = (
        subset
        .groupby("Store")["Sales"]
        .mean()
        .rename(feature)
    )

In [54]:
for feature, values in rolling_windows.items():

    lag_features = lag_features.merge(
        values,
        on="Store",
        how="left",
        validate="one_to_one"
    )

In [55]:
print(
    lag_features.head(10)
)

print("\nMissing historical features:")
print(
    lag_features.isna().sum()
)

   Store  Sales_Lag_1  Sales_Lag_7  Sales_Lag_14  Sales_Rolling_Mean_7  \
0      1         5263         4364          4406           4363.142857   
1      3         8314         3878          4258           6535.714286   
2      7        15344         6574          6861          10394.714286   
3      8         8492         3288          3813           6097.857143   
4      9         8565         5330          5987           6944.285714   
5     10         7185         4713          5761           5280.000000   
6     11        10457         4712          5740           6663.142857   
7     12         8959         5185          7213           7203.571429   
8     13         8821         4361          4956           6031.857143   
9     14         6544         2952          3254           4735.857143   

   Sales_Rolling_Mean_14  Sales_Rolling_Mean_30  
0            3845.714286            3868.100000  
1            5543.000000            5840.800000  
2            8994.214286           

##### Step 23M.10 — Combine everything

In [58]:
# ============================================================
# Step 23M.10 — Build complete Aug 1 feature matrix
# ============================================================

# Test stores
test_stores = test_day[["Store"]].copy()

# ------------------------------------------------------------
# 1. Lag 1 — July 31
# ------------------------------------------------------------

lag_1 = (
    sales_history[
        sales_history["Date"] == pd.Timestamp("2015-07-31")
    ][["Store", "Sales"]]
    .rename(columns={"Sales": "Sales_Lag_1"})
)

# ------------------------------------------------------------
# 2. Lag 7 — July 25
# ------------------------------------------------------------

lag_7 = (
    sales_history[
        sales_history["Date"] == pd.Timestamp("2015-07-25")
    ][["Store", "Sales"]]
    .rename(columns={"Sales": "Sales_Lag_7"})
)

# ------------------------------------------------------------
# 3. Lag 14 — July 18
# ------------------------------------------------------------

lag_14 = (
    sales_history[
        sales_history["Date"] == pd.Timestamp("2015-07-18")
    ][["Store", "Sales"]]
    .rename(columns={"Sales": "Sales_Lag_14"})
)

# ------------------------------------------------------------
# 4. Rolling 7 — July 25 to July 31
# ------------------------------------------------------------

rolling_7 = (
    sales_history[
        (sales_history["Date"] >= pd.Timestamp("2015-07-25"))
        & (sales_history["Date"] <= pd.Timestamp("2015-07-31"))
    ]
    .groupby("Store")["Sales"]
    .mean()
    .rename("Sales_Rolling_Mean_7")
    .reset_index()
)

# ------------------------------------------------------------
# 5. Rolling 14 — July 18 to July 31
# ------------------------------------------------------------

rolling_14 = (
    sales_history[
        (sales_history["Date"] >= pd.Timestamp("2015-07-18"))
        & (sales_history["Date"] <= pd.Timestamp("2015-07-31"))
    ]
    .groupby("Store")["Sales"]
    .mean()
    .rename("Sales_Rolling_Mean_14")
    .reset_index()
)

# ------------------------------------------------------------
# 6. Rolling 30 — July 2 to July 31
# ------------------------------------------------------------

rolling_30 = (
    sales_history[
        (sales_history["Date"] >= pd.Timestamp("2015-07-02"))
        & (sales_history["Date"] <= pd.Timestamp("2015-07-31"))
    ]
    .groupby("Store")["Sales"]
    .mean()
    .rename("Sales_Rolling_Mean_30")
    .reset_index()
)

# ------------------------------------------------------------
# Combine historical features
# ------------------------------------------------------------

historical_features = test_stores.copy()

for feature_df in [
    lag_1,
    lag_7,
    lag_14,
    rolling_7,
    rolling_14,
    rolling_30
]:
    historical_features = historical_features.merge(
        feature_df,
        on="Store",
        how="left",
        validate="one_to_one"
    )

# ------------------------------------------------------------
# Merge with test-day features
# ------------------------------------------------------------

test_day_features = test_day.merge(
    historical_features,
    on="Store",
    how="left",
    validate="one_to_one"
)

print("Historical feature shape:", historical_features.shape)
print("Final test-day shape:", test_day_features.shape)

print("\nHistorical feature columns:")
print(
    historical_features.columns.tolist()
)

print("\nMissing historical features:")
print(
    historical_features[
        [
            "Sales_Lag_1",
            "Sales_Lag_7",
            "Sales_Lag_14",
            "Sales_Rolling_Mean_7",
            "Sales_Rolling_Mean_14",
            "Sales_Rolling_Mean_30"
        ]
    ].isna().sum()
)

Historical feature shape: (856, 7)
Final test-day shape: (856, 33)

Historical feature columns:
['Store', 'Sales_Lag_1', 'Sales_Lag_7', 'Sales_Lag_14', 'Sales_Rolling_Mean_7', 'Sales_Rolling_Mean_14', 'Sales_Rolling_Mean_30']

Missing historical features:
Sales_Lag_1              0
Sales_Lag_7              0
Sales_Lag_14             0
Sales_Rolling_Mean_7     0
Sales_Rolling_Mean_14    0
Sales_Rolling_Mean_30    0
dtype: int64


In [59]:
model_features = [
    "Store",
    "DayOfWeek",
    "Open",
    "Promo",
    "SchoolHoliday",
    "CompetitionDistance",
    "StoreType",
    "Assortment",
    "Year",
    "Month",
    "Day",
    "WeekOfYear",
    "HasCompetitionDistance",
    "HasPromo2",
    "StoreAge",
    "Sales_Lag_1",
    "Sales_Lag_7",
    "Sales_Lag_14",
    "Sales_Rolling_Mean_7",
    "Sales_Rolling_Mean_14",
    "Sales_Rolling_Mean_30"
]

missing = [
    col
    for col in model_features
    if col not in test_day_features.columns
]

print("Missing model features:", missing)

if not missing:
    X_aug1 = test_day_features[model_features].copy()

    print("\nFeature matrix shape:", X_aug1.shape)

    print("\nMissing values:")
    print(
        X_aug1.isna()
        .sum()
        .sort_values(ascending=False)
    )

Missing model features: ['Sales_Lag_1', 'Sales_Lag_7', 'Sales_Lag_14']


##### Step 23M.10A — Inspect the current test-day schema

In [68]:
print("test_day shape:")
print(test_day.shape)

print("\ntest_day columns:")
for i, col in enumerate(test_day.columns, start=1):
    print(f"{i:2}. {col}")

print("\nDuplicate column names:")
duplicates = test_day.columns[
    test_day.columns.duplicated()
].tolist()

print(duplicates)

print("\nHistorical feature columns currently present:")

historical_columns = [
    "Sales_Lag_1",
    "Sales_Lag_7",
    "Sales_Lag_14",
    "Sales_Rolling_Mean_7",
    "Sales_Rolling_Mean_14",
    "Sales_Rolling_Mean_30"
]

for col in historical_columns:
    print(
        f"{col}:",
        col in test_day.columns
    )

test_day shape:
(856, 27)

test_day columns:
 1. Id
 2. Store
 3. DayOfWeek
 4. Date
 5. Open
 6. Promo
 7. StateHoliday
 8. SchoolHoliday
 9. StoreType
10. Assortment
11. CompetitionDistance
12. CompetitionOpenSinceMonth
13. CompetitionOpenSinceYear
14. Promo2
15. Promo2SinceWeek
16. Promo2SinceYear
17. PromoInterval
18. Year
19. Month
20. Day
21. WeekOfYear
22. HasCompetitionDistance
23. HasPromo2
24. StoreAge
25. Sales_Lag_1
26. Sales_Lag_7
27. Sales_Lag_14

Duplicate column names:
[]

Historical feature columns currently present:
Sales_Lag_1: True
Sales_Lag_7: True
Sales_Lag_14: True
Sales_Rolling_Mean_7: False
Sales_Rolling_Mean_14: False
Sales_Rolling_Mean_30: False


##### Step 23M.10B — Verify the existing lag features

In [69]:
lag_columns = [
    "Sales_Lag_1",
    "Sales_Lag_7",
    "Sales_Lag_14"
]

print("Lag feature statistics:")
print(
    test_day[lag_columns].describe()
)

print("\nMissing lag values:")
print(
    test_day[lag_columns].isna().sum()
)

print("\nFirst 10 rows:")
print(
    test_day[
        ["Store", "Date"] + lag_columns
    ].head(10)
)

Lag feature statistics:
        Sales_Lag_1   Sales_Lag_7  Sales_Lag_14
count    856.000000    856.000000    856.000000
mean    9062.334112   5598.831776   6014.400701
std     3014.859679   2497.198638   2513.821264
min     2726.000000    904.000000   1100.000000
25%     6991.500000   3898.000000   4369.750000
50%     8570.500000   5283.000000   5653.000000
75%    10560.500000   6859.750000   7416.250000
max    27508.000000  23091.000000  22836.000000

Missing lag values:
Sales_Lag_1     0
Sales_Lag_7     0
Sales_Lag_14    0
dtype: int64

First 10 rows:
   Store       Date  Sales_Lag_1  Sales_Lag_7  Sales_Lag_14
0      1 2015-08-01         5263         4364          4406
1      3 2015-08-01         8314         3878          4258
2      7 2015-08-01        15344         6574          6861
3      8 2015-08-01         8492         3288          3813
4      9 2015-08-01         8565         5330          5987
5     10 2015-08-01         7185         4713          5761
6     11 2015-08-01 

##### Step 23M.10C — Create the Rolling Features

In [70]:
# Make sure dates are datetime
sales_history["Date"] = pd.to_datetime(
    sales_history["Date"]
)

# ------------------------------------------------------------
# Rolling 7-day mean
# Jul 25 → Jul 31
# ------------------------------------------------------------

rolling_7 = (
    sales_history[
        (sales_history["Date"] >= pd.Timestamp("2015-07-25"))
        & (sales_history["Date"] <= pd.Timestamp("2015-07-31"))
    ]
    .groupby("Store")["Sales"]
    .mean()
    .rename("Sales_Rolling_Mean_7")
    .reset_index()
)

# ------------------------------------------------------------
# Rolling 14-day mean
# Jul 18 → Jul 31
# ------------------------------------------------------------

rolling_14 = (
    sales_history[
        (sales_history["Date"] >= pd.Timestamp("2015-07-18"))
        & (sales_history["Date"] <= pd.Timestamp("2015-07-31"))
    ]
    .groupby("Store")["Sales"]
    .mean()
    .rename("Sales_Rolling_Mean_14")
    .reset_index()
)

# ------------------------------------------------------------
# Rolling 30-day mean
# Jul 2 → Jul 31
# ------------------------------------------------------------

rolling_30 = (
    sales_history[
        (sales_history["Date"] >= pd.Timestamp("2015-07-02"))
        & (sales_history["Date"] <= pd.Timestamp("2015-07-31"))
    ]
    .groupby("Store")["Sales"]
    .mean()
    .rename("Sales_Rolling_Mean_30")
    .reset_index()
)

# ------------------------------------------------------------
# Display validation
# ------------------------------------------------------------

print("Rolling 7 shape:", rolling_7.shape)
print("Rolling 14 shape:", rolling_14.shape)
print("Rolling 30 shape:", rolling_30.shape)

print("\nMissing values:")

print(
    "Rolling 7:",
    rolling_7["Sales_Rolling_Mean_7"].isna().sum()
)

print(
    "Rolling 14:",
    rolling_14["Sales_Rolling_Mean_14"].isna().sum()
)

print(
    "Rolling 30:",
    rolling_30["Sales_Rolling_Mean_30"].isna().sum()
)

print("\nFirst 10 rolling values:")

print(
    rolling_7.head(10)
)

print(
    rolling_14.head(10)
)

print(
    rolling_30.head(10)
)

Rolling 7 shape: (1115, 2)
Rolling 14 shape: (1115, 2)
Rolling 30 shape: (1115, 2)

Missing values:
Rolling 7: 0
Rolling 14: 0
Rolling 30: 0

First 10 rolling values:
   Store  Sales_Rolling_Mean_7
0      1           4363.142857
1      2           4691.857143
2      3           6535.714286
3      4           9472.142857
4      5           4405.142857
5      6           4167.142857
6      7          10394.714286
7      8           6097.857143
8      9           6944.285714
9     10           5280.000000
   Store  Sales_Rolling_Mean_14
0      1            3845.714286
1      2            4113.285714
2      3            5543.000000
3      4            8541.285714
4      5            3696.214286
5      6            3632.928571
6      7            8994.214286
7      8            5260.285714
8      9            6086.142857
9     10            5244.642857
   Store  Sales_Rolling_Mean_30
0      1            3868.100000
1      2            4244.466667
2      3            5840.800000
3      4    

##### Step 23M.10D — Add rolling features to test_day

In [73]:

rolling_features = rolling_7.merge(
    rolling_14,
    on="Store",
    how="inner"
).merge(
    rolling_30,
    on="Store",
    how="inner"
)

print("Rolling feature table shape:")
print(rolling_features.shape)

print("\nDuplicate Store IDs:")
print(
    rolling_features["Store"].duplicated().sum()
)

print("\nRolling feature columns:")
print(
    rolling_features.columns.tolist()
)

# ------------------------------------------------------------
# Keep only the required rolling columns
# ------------------------------------------------------------

rolling_columns = [
    "Store",
    "Sales_Rolling_Mean_7",
    "Sales_Rolling_Mean_14",
    "Sales_Rolling_Mean_30"
]

rolling_features = rolling_features[
    rolling_columns
].copy()

# ------------------------------------------------------------
# Merge onto test_day
# ------------------------------------------------------------

test_day = test_day.merge(
    rolling_features,
    on="Store",
    how="left",
    validate="one_to_one"
)

print("\nFinal test_day shape:")
print(test_day.shape)

print("\nRolling features now present:")

for col in rolling_columns[1:]:
    print(
        f"{col}:",
        col in test_day.columns
    )

print("\nMissing rolling values:")

print(
    test_day[
        rolling_columns[1:]
    ].isna().sum()
)

Rolling feature table shape:
(1115, 4)

Duplicate Store IDs:
0

Rolling feature columns:
['Store', 'Sales_Rolling_Mean_7', 'Sales_Rolling_Mean_14', 'Sales_Rolling_Mean_30']

Final test_day shape:
(856, 36)

Rolling features now present:
Sales_Rolling_Mean_7: True
Sales_Rolling_Mean_14: True
Sales_Rolling_Mean_30: True

Missing rolling values:
Sales_Rolling_Mean_7     0
Sales_Rolling_Mean_14    0
Sales_Rolling_Mean_30    0
dtype: int64


##### Step 23M.10E — Build the final 21-feature test matrix

In [75]:
model_features = [
    "Store",
    "DayOfWeek",
    "Open",
    "Promo",
    "SchoolHoliday",
    "CompetitionDistance",
    "StoreType",
    "Assortment",
    "Year",
    "Month",
    "Day",
    "WeekOfYear",
    "HasCompetitionDistance",
    "HasPromo2",
    "StoreAge",
    "Sales_Lag_1",
    "Sales_Lag_7",
    "Sales_Lag_14",
    "Sales_Rolling_Mean_7",
    "Sales_Rolling_Mean_14",
    "Sales_Rolling_Mean_30",
]

print("Number of model features:", len(model_features))

print("\nChecking model features...")

missing_features = [
    col
    for col in model_features
    if col not in test_day.columns
]

print("Missing features:", missing_features)

assert not missing_features, (
    f"Missing model features: {missing_features}"
)

# ------------------------------------------------------------
# Create clean final feature matrix
# ------------------------------------------------------------

X_test_final = test_day[
    model_features
].copy()

print("\nFinal test feature shape:")
print(X_test_final.shape)

print("\nFinal test feature columns:")

for i, col in enumerate(
    X_test_final.columns,
    start=1
):
    print(f"{i:2}. {col}")

# ------------------------------------------------------------
# Check missing values
# ------------------------------------------------------------

print("\nMissing values:")

missing_values = (
    X_test_final
    .isna()
    .sum()
)

print(
    missing_values[
        missing_values > 0
    ]
)

# ------------------------------------------------------------
# Check row count
# ------------------------------------------------------------

print("\nExpected rows:", 856)
print("Actual rows:", len(X_test_final))

assert len(X_test_final) == 856

# ------------------------------------------------------------
# Check feature count
# ------------------------------------------------------------

assert X_test_final.shape[1] == 21

print("\n✓ Final 21-feature test matrix created successfully")

Number of model features: 21

Checking model features...
Missing features: []

Final test feature shape:
(856, 21)

Final test feature columns:
 1. Store
 2. DayOfWeek
 3. Open
 4. Promo
 5. SchoolHoliday
 6. CompetitionDistance
 7. StoreType
 8. Assortment
 9. Year
10. Month
11. Day
12. WeekOfYear
13. HasCompetitionDistance
14. HasPromo2
15. StoreAge
16. Sales_Lag_1
17. Sales_Lag_7
18. Sales_Lag_14
19. Sales_Rolling_Mean_7
20. Sales_Rolling_Mean_14
21. Sales_Rolling_Mean_30

Missing values:
CompetitionDistance      2
StoreAge               317
dtype: int64

Expected rows: 856
Actual rows: 856

✓ Final 21-feature test matrix created successfully


##### Step 23M.10F — Investigate the remaining missing values

In [79]:
from pathlib import Path
import pandas as pd

# Notebook is inside:
# E:\Praxis\rossmann-sales\notebooks
#
# Therefore project root is one level above cwd.

PROJECT_ROOT = Path.cwd().parent

TRAIN_FEATURES_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "train_features.csv"
)

print("Project root:")
print(PROJECT_ROOT)

print("\nTraining feature path:")
print(TRAIN_FEATURES_PATH)

print("\nFile exists:")
print(TRAIN_FEATURES_PATH.exists())

assert TRAIN_FEATURES_PATH.exists(), (
    f"Training feature file not found: {TRAIN_FEATURES_PATH}"
)

# ------------------------------------------------------------
# Load training features
# ------------------------------------------------------------

train_features = pd.read_csv(
    TRAIN_FEATURES_PATH,
    low_memory=False
)

print("\nTraining feature shape:")
print(train_features.shape)


# ------------------------------------------------------------
# Investigate missing test values
# ------------------------------------------------------------

missing_cols = [
    "CompetitionDistance",
    "StoreAge"
]

print("\n" + "=" * 60)
print("TEST MISSING-VALUE INVESTIGATION")
print("=" * 60)

for col in missing_cols:

    print(f"\n{col}")
    print("-" * 60)

    missing_rows = test_day[
        test_day[col].isna()
    ][
        [
            "Store",
            "Date",
            "CompetitionDistance",
            "CompetitionOpenSinceMonth",
            "CompetitionOpenSinceYear",
            "StoreAge"
        ]
    ]

    print("Missing rows:", len(missing_rows))
    print(missing_rows.head(20).to_string(index=False))


# ------------------------------------------------------------
# Compare training missing values
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("TRAINING FEATURE MISSING VALUES")
print("=" * 60)

for col in missing_cols:

    print(f"\n{col}")
    print("-" * 60)

    print(
        "Missing:",
        train_features[col].isna().sum()
    )

    print(
        train_features[col].describe()
    )


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)

for col in missing_cols:

    train_missing = train_features[col].isna().sum()
    test_missing = test_day[col].isna().sum()

    print(
        f"{col}: "
        f"train missing = {train_missing}, "
        f"test missing = {test_missing}"
    )

Project root:
e:\Praxis\rossmann-sales

Training feature path:
e:\Praxis\rossmann-sales\data\processed\train_features.csv

File exists:
True

Training feature shape:
(1017209, 31)

TEST MISSING-VALUE INVESTIGATION

CompetitionDistance
------------------------------------------------------------
Missing rows: 2
 Store       Date  CompetitionDistance  CompetitionOpenSinceMonth  CompetitionOpenSinceYear  StoreAge
   622 2015-08-01                  NaN                        NaN                       NaN       NaN
   879 2015-08-01                  NaN                        NaN                       NaN       NaN

StoreAge
------------------------------------------------------------
Missing rows: 317
 Store       Date  CompetitionDistance  CompetitionOpenSinceMonth  CompetitionOpenSinceYear  StoreAge
    12 2015-08-01               1070.0                        NaN                       NaN       NaN
    13 2015-08-01                310.0                        NaN                       N

##### Step 23M.10G — Audit the training preprocessing

In [81]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

RF_PATH = (
    PROJECT_ROOT
    / "src"
    / "models"
    / "train_random_forest.py"
)

print("Training script:")
print(RF_PATH)

print("\nFile exists:")
print(RF_PATH.exists())

assert RF_PATH.exists()

print("\n" + "=" * 70)
print("RELEVANT PREPROCESSING CODE")
print("=" * 70)

code = RF_PATH.read_text(
    encoding="utf-8"
)

lines = code.splitlines()

keywords = [
    "fillna",
    "SimpleImputer",
    "Imputer",
    "OneHotEncoder",
    "ColumnTransformer",
    "OrdinalEncoder",
    "get_dummies",
    "missing",
    "nan",
    "NaN",
    "CompetitionDistance",
    "StoreAge",
]

for i, line in enumerate(lines, start=1):

    if any(
        keyword.lower() in line.lower()
        for keyword in keywords
    ):
        start = max(0, i - 3)
        end = min(len(lines), i + 3)

        print(
            f"\n--- Lines {start + 1}-{end} ---"
        )

        for j in range(start, end):
            print(
                f"{j + 1:4}: {lines[j]}"
            )

Training script:
e:\Praxis\rossmann-sales\src\models\train_random_forest.py

File exists:
True

RELEVANT PREPROCESSING CODE

--- Lines 56-61 ---
  56:         "Promo",
  57:         "SchoolHoliday",
  58:         "CompetitionDistance",
  59:         "StoreType",
  60:         "Assortment",
  61:         "Year",

--- Lines 63-68 ---
  63:         "Day",
  64:         "WeekOfYear",
  65:         "HasCompetitionDistance",
  66:         "HasPromo2",
  67:         "StoreAge",
  68:         "Sales_Lag_1",

--- Lines 65-70 ---
  65:         "HasCompetitionDistance",
  66:         "HasPromo2",
  67:         "StoreAge",
  68:         "Sales_Lag_1",
  69:         "Sales_Lag_7",
  70:         "Sales_Lag_14",

--- Lines 89-94 ---
  89:     ]
  90: 
  91:     X_train = pd.get_dummies(
  92:         X_train,
  93:         columns=categorical_columns,
  94:         dtype=int

--- Lines 95-100 ---
  95:     )
  96: 
  97:     X_validation = pd.get_dummies(
  98:         X_validation,
  99:         col

##### Step 23M.10H — Reproduce model preprocessing on test data

In [83]:
import pandas as pd

# ------------------------------------------------------------
# Model feature definition
# ------------------------------------------------------------

model_features = [
    "Store",
    "DayOfWeek",
    "Open",
    "Promo",
    "SchoolHoliday",
    "CompetitionDistance",
    "StoreType",
    "Assortment",
    "Year",
    "Month",
    "Day",
    "WeekOfYear",
    "HasCompetitionDistance",
    "HasPromo2",
    "StoreAge",
    "Sales_Lag_1",
    "Sales_Lag_7",
    "Sales_Lag_14",
    "Sales_Rolling_Mean_7",
    "Sales_Rolling_Mean_14",
    "Sales_Rolling_Mean_30",
]

categorical_columns = [
    "StoreType",
    "Assortment",
]

# ------------------------------------------------------------
# Select exactly the raw model features
# ------------------------------------------------------------

X_test_raw = test_day[
    model_features
].copy()

print("Raw test matrix:")
print(X_test_raw.shape)

# ------------------------------------------------------------
# Check missing values BEFORE preprocessing
# ------------------------------------------------------------

print("\nMissing values before preprocessing:")

missing_before = X_test_raw.isna().sum()

print(
    missing_before[
        missing_before > 0
    ]
)

# ------------------------------------------------------------
# Apply EXACT training preprocessing
# ------------------------------------------------------------

X_test_processed = pd.get_dummies(
    X_test_raw,
    columns=categorical_columns,
    dtype=int
)

# Same missing-value treatment as training
X_test_processed = X_test_processed.fillna(0)

print("\nProcessed test matrix:")
print(X_test_processed.shape)

# ------------------------------------------------------------
# Check missing values AFTER preprocessing
# ------------------------------------------------------------

print("\nMissing values after preprocessing:")

missing_after = X_test_processed.isna().sum()

print(
    missing_after[
        missing_after > 0
    ]
)

# ------------------------------------------------------------
# Display columns
# ------------------------------------------------------------

print("\nProcessed test columns:")

for i, col in enumerate(
    X_test_processed.columns,
    start=1
):
    print(f"{i:2}. {col}")

print("\n✓ Test preprocessing completed")

Raw test matrix:
(856, 21)

Missing values before preprocessing:
CompetitionDistance      2
StoreAge               317
dtype: int64

Processed test matrix:
(856, 26)

Missing values after preprocessing:
Series([], dtype: int64)

Processed test columns:
 1. Store
 2. DayOfWeek
 3. Open
 4. Promo
 5. SchoolHoliday
 6. CompetitionDistance
 7. Year
 8. Month
 9. Day
10. WeekOfYear
11. HasCompetitionDistance
12. HasPromo2
13. StoreAge
14. Sales_Lag_1
15. Sales_Lag_7
16. Sales_Lag_14
17. Sales_Rolling_Mean_7
18. Sales_Rolling_Mean_14
19. Sales_Rolling_Mean_30
20. StoreType_a
21. StoreType_b
22. StoreType_c
23. StoreType_d
24. Assortment_a
25. Assortment_b
26. Assortment_c

✓ Test preprocessing completed


##### Step 23M.10I — Verify exact trained-model schema

In [84]:
from pathlib import Path
import joblib

PROJECT_ROOT = Path.cwd().parent

MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "random_forest_final.pkl"
)

print("Model path:")
print(MODEL_PATH)

print("\nModel exists:")
print(MODEL_PATH.exists())

assert MODEL_PATH.exists(), (
    f"Model not found: {MODEL_PATH}"
)

# ------------------------------------------------------------
# Load model
# ------------------------------------------------------------

model = joblib.load(
    MODEL_PATH
)

print("\nModel type:")
print(type(model))

# ------------------------------------------------------------
# Inspect expected feature count
# ------------------------------------------------------------

print("\nExpected feature count:")

if hasattr(model, "n_features_in_"):
    print(model.n_features_in_)
else:
    print("n_features_in_ not available")


# ------------------------------------------------------------
# Compare with test matrix
# ------------------------------------------------------------

print("\nTest feature count:")
print(X_test_processed.shape[1])

print("\nFeature count match:")

if hasattr(model, "n_features_in_"):

    print(
        model.n_features_in_
        == X_test_processed.shape[1]
    )


# ------------------------------------------------------------
# Inspect feature names if available
# ------------------------------------------------------------

if hasattr(model, "feature_names_in_"):

    model_features_actual = list(
        model.feature_names_in_
    )

    test_features_actual = list(
        X_test_processed.columns
    )

    print("\nModel feature names:")
    for i, col in enumerate(
        model_features_actual,
        start=1
    ):
        print(f"{i:2}. {col}")

    print("\n" + "=" * 60)
    print("EXACT FEATURE ORDER MATCH")
    print("=" * 60)

    exact_match = (
        model_features_actual
        == test_features_actual
    )

    print(
        "Exact match:",
        exact_match
    )

    if not exact_match:

        print("\nColumns in model but not test:")

        print(
            sorted(
                set(model_features_actual)
                - set(test_features_actual)
            )
        )

        print("\nColumns in test but not model:")

        print(
            sorted(
                set(test_features_actual)
                - set(model_features_actual)
            )
        )

else:

    print(
        "\nModel does not expose feature_names_in_."
    )

Model path:
e:\Praxis\rossmann-sales\models\random_forest_final.pkl

Model exists:
True


c:\Users\HP\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.9.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(



Model type:
<class 'dict'>

Expected feature count:
n_features_in_ not available

Test feature count:
26

Feature count match:

Model does not expose feature_names_in_.


c:\Users\HP\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator RandomForestRegressor from version 1.9.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


##### Step 23M.10J — Inspect the model package

In [87]:
# ============================================================
#   Check installed ML versions
# ============================================================

import sklearn
import pandas
import numpy
import joblib

print("Environment versions")
print("=" * 40)

print("Python:")
import sys
print(sys.version)

print("\nscikit-learn:")
print(sklearn.__version__)

print("\npandas:")
print(pandas.__version__)

print("\nnumpy:")
print(numpy.__version__)

print("\njoblib:")
print(joblib.__version__)

Environment versions
Python:
3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]

scikit-learn:
1.8.0

pandas:
2.3.3

numpy:
2.4.2

joblib:
1.5.3


In [88]:
import joblib
import sklearn

print("Current scikit-learn version:")
print(sklearn.__version__)

print("\nLoading final model package...")

model_package = joblib.load(
    MODEL_PATH
)

print("\nModel package type:")
print(type(model_package))

print("\nModel package keys:")

if isinstance(model_package, dict):

    for key in model_package.keys():
        print("-", key)

else:

    print("Not a dictionary")


# ------------------------------------------------------------
# Inspect each package item
# ------------------------------------------------------------

if isinstance(model_package, dict):

    print("\n" + "=" * 60)
    print("PACKAGE CONTENTS")
    print("=" * 60)

    for key, value in model_package.items():

        print(f"\n{key}")
        print("-" * 60)

        print("Type:", type(value))

        if hasattr(value, "n_features_in_"):
            print(
                "n_features_in_:",
                value.n_features_in_
            )

        if hasattr(value, "feature_names_in_"):
            print(
                "feature_names_in_:",
                list(value.feature_names_in_)
            )

        if isinstance(value, (list, tuple)):
            print(
                "Length:",
                len(value)
            )

            if len(value) <= 50:
                print(
                    "Values:",
                    value
                )

        elif isinstance(value, dict):
            print(
                "Nested keys:",
                list(value.keys())
            )

Current scikit-learn version:
1.8.0

Loading final model package...

Model package type:
<class 'dict'>

Model package keys:
- model
- features

PACKAGE CONTENTS

model
------------------------------------------------------------
Type: <class 'sklearn.ensemble._forest.RandomForestRegressor'>
n_features_in_: 26
feature_names_in_: ['Store', 'DayOfWeek', 'Open', 'Promo', 'SchoolHoliday', 'CompetitionDistance', 'Year', 'Month', 'Day', 'WeekOfYear', 'HasCompetitionDistance', 'HasPromo2', 'StoreAge', 'Sales_Lag_1', 'Sales_Lag_7', 'Sales_Lag_14', 'Sales_Rolling_Mean_7', 'Sales_Rolling_Mean_14', 'Sales_Rolling_Mean_30', 'StoreType_a', 'StoreType_b', 'StoreType_c', 'StoreType_d', 'Assortment_a', 'Assortment_b', 'Assortment_c']

features
------------------------------------------------------------
Type: <class 'list'>
Length: 26
Values: ['Store', 'DayOfWeek', 'Open', 'Promo', 'SchoolHoliday', 'CompetitionDistance', 'Year', 'Month', 'Day', 'WeekOfYear', 'HasCompetitionDistance', 'HasPromo2', 'Sto

In [89]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

VALIDATION_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "validation_features.csv"
)

print("Validation path:")
print(VALIDATION_PATH)

print("\nExists:")
print(VALIDATION_PATH.exists())

if VALIDATION_PATH.exists():

    validation_features = pd.read_csv(
        VALIDATION_PATH,
        low_memory=False
    )

    print("\nShape:")
    print(validation_features.shape)

    print("\nDate range:")
    print(validation_features["Date"].min())
    print(validation_features["Date"].max())

    print("\nColumns:")
    print(validation_features.columns.tolist())

    print("\nHistorical feature missing values:")
    print(
        validation_features[
            [
                "Sales_Lag_1",
                "Sales_Lag_7",
                "Sales_Lag_14",
                "Sales_Rolling_Mean_7",
                "Sales_Rolling_Mean_14",
                "Sales_Rolling_Mean_30",
            ]
        ].isna().sum()
    )

Validation path:
e:\Praxis\rossmann-sales\data\processed\validation_features.csv

Exists:
False


##### Step 24B — inspect the actual training/validation split

In [92]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

TRAIN_SPLIT_PATH = (
    PROJECT_ROOT
    / "data"
    / "splits"
    / "train.csv"
)

VALIDATION_SPLIT_PATH = (
    PROJECT_ROOT
    / "data"
    / "splits"
    / "validation.csv"
)

print("Project root:")
print(PROJECT_ROOT)

print("\nTrain split:")
print(TRAIN_SPLIT_PATH)
print("Exists:", TRAIN_SPLIT_PATH.exists())

print("\nValidation split:")
print(VALIDATION_SPLIT_PATH)
print("Exists:", VALIDATION_SPLIT_PATH.exists())

Project root:
e:\Praxis\rossmann-sales

Train split:
e:\Praxis\rossmann-sales\data\splits\train.csv
Exists: True

Validation split:
e:\Praxis\rossmann-sales\data\splits\validation.csv
Exists: True


In [93]:
train_split = pd.read_csv(
    TRAIN_SPLIT_PATH,
    low_memory=False
)

validation_split = pd.read_csv(
    VALIDATION_SPLIT_PATH,
    low_memory=False
)

print("=" * 70)
print("TRAIN SPLIT")
print("=" * 70)

print("Shape:", train_split.shape)
print("Columns:")
print(train_split.columns.tolist())

print("\nDate range:")
print(train_split["Date"].min(), "→", train_split["Date"].max())

print("\nSales present:", "Sales" in train_split.columns)

print("\n" + "=" * 70)
print("VALIDATION SPLIT")
print("=" * 70)

print("Shape:", validation_split.shape)
print("Columns:")
print(validation_split.columns.tolist())

print("\nDate range:")
print(validation_split["Date"].min(), "→", validation_split["Date"].max())

print("\nSales present:", "Sales" in validation_split.columns)

TRAIN SPLIT
Shape: (780829, 31)
Columns:
['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval', 'Year', 'Month', 'Day', 'WeekOfYear', 'HasCompetitionDistance', 'HasPromo2', 'StoreAge', 'Sales_Lag_1', 'Sales_Lag_7', 'Sales_Lag_14', 'Sales_Rolling_Mean_7', 'Sales_Rolling_Mean_14', 'Sales_Rolling_Mean_30']

Date range:
2013-01-01 → 2014-12-31

Sales present: True

VALIDATION SPLIT
Shape: (201815, 31)
Columns:
['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval', 'Year', 'Month', 'Day', 'WeekOfYear', 'HasCompetitionDistance', 'HasPromo2', 'StoreAge', 'Sales

In [94]:
print("=" * 70)
print("VALIDATION DATE COVERAGE")
print("=" * 70)

validation_split["Date"] = pd.to_datetime(validation_split["Date"])

daily_counts = (
    validation_split
    .groupby("Date")["Store"]
    .nunique()
)

print(daily_counts.head(10))
print("\n...")
print(daily_counts.tail(10))

print("\nUnique stores per validation day:")
print(daily_counts.unique())

VALIDATION DATE COVERAGE
Date
2015-01-01    1115
2015-01-02    1115
2015-01-03    1115
2015-01-04    1115
2015-01-05    1115
2015-01-06    1115
2015-01-07    1115
2015-01-08    1115
2015-01-09    1115
2015-01-10    1115
Name: Store, dtype: int64

...
Date
2015-06-21    1115
2015-06-22    1115
2015-06-23    1115
2015-06-24    1115
2015-06-25    1115
2015-06-26    1115
2015-06-27    1115
2015-06-28    1115
2015-06-29    1115
2015-06-30    1115
Name: Store, dtype: int64

Unique stores per validation day:
[1115]


##### Step 24C — find how the splits were created

In [95]:
print("\nSearching for split creation code...")
print("=" * 70)

search_terms = [
    "train.csv",
    "validation.csv",
    "train_test_split",
    "split",
    "2014-12-31",
    "2015-01-01",
    "Sales_Lag_1",
    "Sales_Rolling_Mean_7",
]

for path in sorted((PROJECT_ROOT / "src").rglob("*.py")):

    try:
        text = path.read_text(encoding="utf-8")
    except Exception:
        continue

    matches = [
        term
        for term in search_terms
        if term in text
    ]

    if matches:
        print(f"\n{path.relative_to(PROJECT_ROOT)}")
        print("Matches:", matches)


Searching for split creation code...

src\data\make_dataset.py
Matches: ['train.csv']

src\data\preprocess_data.py
Matches: ['train.csv']

src\data\split_data.py
Matches: ['train.csv', 'validation.csv', 'split', '2014-12-31', '2015-01-01']

src\data\validate_data.py
Matches: ['train.csv']

src\features\build_features.py
Matches: ['Sales_Lag_1', 'Sales_Rolling_Mean_7']

src\models\baseline.py
Matches: ['validation.csv', 'split']

src\models\evaluate_final_model.py
Matches: ['split', 'Sales_Lag_1', 'Sales_Rolling_Mean_7']

src\models\mlflow_tracking.py
Matches: ['train.csv', 'validation.csv', 'split', 'Sales_Lag_1', 'Sales_Rolling_Mean_7']

src\models\predict.py
Matches: ['split', 'Sales_Lag_1', 'Sales_Rolling_Mean_7']

src\models\predict_production.py
Matches: ['split', 'Sales_Lag_1', 'Sales_Rolling_Mean_7']

src\models\train_final_model.py
Matches: ['train.csv', 'split', 'Sales_Lag_1', 'Sales_Rolling_Mean_7']

src\models\train_random_forest.py
Matches: ['train.csv', 'validation.csv', 

##### Step 24D — inspect split_data.py

In [96]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

SPLIT_SCRIPT = (
    PROJECT_ROOT
    / "src"
    / "data"
    / "split_data.py"
)

print("=" * 70)
print("split_data.py")
print("=" * 70)

with open(SPLIT_SCRIPT, "r", encoding="utf-8") as f:
    lines = f.readlines()

for i, line in enumerate(lines, start=1):
    print(f"{i:4}: {line.rstrip()}")

split_data.py
   1: from pathlib import Path
   2: import pandas as pd
   3: 
   4: 
   5: PROJECT_ROOT = Path(__file__).resolve().parents[2]
   6: 
   7: INPUT_PATH = (
   8:     PROJECT_ROOT
   9:     / "data"
  10:     / "processed"
  11:     / "train_features.csv"
  12: )
  13: 
  14: OUTPUT_DIR = (
  15:     PROJECT_ROOT
  16:     / "data"
  17:     / "splits"
  18: )
  19: 
  20: 
  21: def main():
  22: 
  23:     print("Loading feature dataset...")
  24: 
  25:     df = pd.read_csv(
  26:         INPUT_PATH,
  27:         low_memory=False,
  28:         dtype={"StateHoliday": "string"}
  29:     )
  30: 
  31:     df["Date"] = pd.to_datetime(df["Date"])
  32: 
  33:     print(f"Dataset shape: {df.shape}")
  34:     print(
  35:         f"Date range: {df['Date'].min().date()} "
  36:         f"to {df['Date'].max().date()}"
  37:     )
  38: 
  39:     # -----------------------------
  40:     # Time-based split
  41:     # -----------------------------
  42: 
  43:     train = d

In [97]:
FEATURE_SCRIPT = (
    PROJECT_ROOT
    / "src"
    / "features"
    / "build_features.py"
)

print("=" * 70)
print("build_features.py")
print("=" * 70)

with open(FEATURE_SCRIPT, "r", encoding="utf-8") as f:
    lines = f.readlines()

for i, line in enumerate(lines, start=1):
    print(f"{i:4}: {line.rstrip()}")

build_features.py
   1: from pathlib import Path
   2: 
   3: import pandas as pd
   4: 
   5: 
   6: # --------------------------------------------------
   7: # Project paths
   8: # --------------------------------------------------
   9: 
  10: PROJECT_ROOT = Path(__file__).resolve().parents[2]
  11: 
  12: INPUT_PATH = (
  13:     PROJECT_ROOT
  14:     / "data"
  15:     / "processed"
  16:     / "train_processed.csv"
  17: )
  18: 
  19: OUTPUT_DIR = (
  20:     PROJECT_ROOT
  21:     / "data"
  22:     / "processed"
  23: )
  24: 
  25: OUTPUT_PATH = (
  26:     OUTPUT_DIR
  27:     / "train_features.csv"
  28: )
  29: 
  30: 
  31: # --------------------------------------------------
  32: # Load processed data
  33: # --------------------------------------------------
  34: 
  35: def load_data():
  36: 
  37:     print("Loading processed dataset...")
  38: 
  39:     df = pd.read_csv(
  40:         INPUT_PATH,
  41:         low_memory=False,
  42:         dtype={"StateHolida

##### Step 24E — inspect July holdout

In [98]:
print("=" * 70)
print("JULY 2015 HOLDOUT INSPECTION")
print("=" * 70)

july = pd.read_csv(
    PROJECT_ROOT / "data" / "splits" / "test.csv",
    low_memory=False
)

july["Date"] = pd.to_datetime(july["Date"])

print("Shape:", july.shape)

print("\nColumns:")
print(july.columns.tolist())

print("\nDate range:")
print(july["Date"].min(), "→", july["Date"].max())

print("\nRows per day:")
print(
    july.groupby("Date")["Store"]
    .nunique()
    .value_counts()
)

print("\nUnique stores:")
print(july["Store"].nunique())

print("\nSales present:")
print("Sales" in july.columns)

JULY 2015 HOLDOUT INSPECTION
Shape: (34565, 31)

Columns:
['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval', 'Year', 'Month', 'Day', 'WeekOfYear', 'HasCompetitionDistance', 'HasPromo2', 'StoreAge', 'Sales_Lag_1', 'Sales_Lag_7', 'Sales_Lag_14', 'Sales_Rolling_Mean_7', 'Sales_Rolling_Mean_14', 'Sales_Rolling_Mean_30']

Date range:
2015-07-01 00:00:00 → 2015-07-31 00:00:00

Rows per day:
Store
1115    31
Name: count, dtype: int64

Unique stores:
1115

Sales present:
True


In [99]:
print("=" * 70)
print("JULY HISTORICAL FEATURES")
print("=" * 70)

historical_columns = [
    "Sales_Lag_1",
    "Sales_Lag_7",
    "Sales_Lag_14",
    "Sales_Rolling_Mean_7",
    "Sales_Rolling_Mean_14",
    "Sales_Rolling_Mean_30",
]

for col in historical_columns:
    print(
        f"{col}:",
        col in july.columns
    )

JULY HISTORICAL FEATURES
Sales_Lag_1: True
Sales_Lag_7: True
Sales_Lag_14: True
Sales_Rolling_Mean_7: True
Sales_Rolling_Mean_14: True
Sales_Rolling_Mean_30: True


In [100]:
print("=" * 70)
print("JULY FIRST DAY FEATURES")
print("=" * 70)

july_first = (
    july[july["Date"] == "2015-07-01"]
    .copy()
)

print("Shape:", july_first.shape)

print(
    july_first[
        [
            "Store",
            "Date",
            "Sales",
            "Sales_Lag_1",
            "Sales_Lag_7",
            "Sales_Lag_14",
            "Sales_Rolling_Mean_7",
            "Sales_Rolling_Mean_14",
            "Sales_Rolling_Mean_30",
        ]
    ].head(10)
)

JULY FIRST DAY FEATURES
Shape: (1115, 31)
     Store       Date  Sales  Sales_Lag_1  Sales_Lag_7  Sales_Lag_14  \
0        1 2015-07-01   5223       5735.0       3346.0        4000.0   
31       2 2015-07-01   6431       9863.0       4805.0        6924.0   
62       3 2015-07-01   8311      13261.0       5448.0        8524.0   
93       4 2015-07-01  11216      13106.0       8048.0        9685.0   
124      5 2015-07-01   6253       6635.0       4134.0        5632.0   
155      6 2015-07-01   5804       7427.0       3970.0        5135.0   
186      7 2015-07-01  11024      14838.0       6913.0        9041.0   
217      8 2015-07-01   8523       9836.0       5005.0        7751.0   
248      9 2015-07-01   8490      10447.0       6888.0        8045.0   
279     10 2015-07-01   7896       8565.0       4896.0        6338.0   

     Sales_Rolling_Mean_7  Sales_Rolling_Mean_14  Sales_Rolling_Mean_30  
0             3592.428571            3549.928571            3667.033333  
31            471

##### Step 23M.10B — Validate train/model/test feature schema

In [102]:
# ============================================================
# Step 23M.10B — Reproduce model preprocessing exactly
# ============================================================

from pathlib import Path
import pandas as pd
import joblib


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


TRAIN_FEATURES_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "train_features.csv"
)

MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "random_forest_final.pkl"
)


print("=" * 70)
print("STEP 23M.10B — MODEL PREPROCESSING VALIDATION")
print("=" * 70)


# ------------------------------------------------------------
# Load training features
# ------------------------------------------------------------

train_features = pd.read_csv(
    TRAIN_FEATURES_PATH,
    low_memory=False
)

print("\nTraining feature dataset:")
print(train_features.shape)


# ------------------------------------------------------------
# Load model package
# ------------------------------------------------------------

model_package = joblib.load(MODEL_PATH)

model = model_package["model"]
model_features = model_package["features"]


print("\nModel type:")
print(type(model))

print("\nModel feature count:")
print(len(model_features))


# ------------------------------------------------------------
# Original model features BEFORE one-hot encoding
# ------------------------------------------------------------

base_features = [
    "Store",
    "DayOfWeek",
    "Open",
    "Promo",
    "SchoolHoliday",
    "CompetitionDistance",
    "StoreType",
    "Assortment",
    "Year",
    "Month",
    "Day",
    "WeekOfYear",
    "HasCompetitionDistance",
    "HasPromo2",
    "StoreAge",
    "Sales_Lag_1",
    "Sales_Lag_7",
    "Sales_Lag_14",
    "Sales_Rolling_Mean_7",
    "Sales_Rolling_Mean_14",
    "Sales_Rolling_Mean_30",
]


# ------------------------------------------------------------
# Verify base features
# ------------------------------------------------------------

missing_base = [
    col
    for col in base_features
    if col not in train_features.columns
]

print("\nMissing base features:")
print(missing_base)

assert not missing_base, (
    f"Missing base features: {missing_base}"
)


# ------------------------------------------------------------
# Create training matrix
# EXACTLY like train_random_forest.py
# ------------------------------------------------------------

X_train = train_features[base_features].copy()


categorical_columns = [
    "StoreType",
    "Assortment",
]


X_train = pd.get_dummies(
    X_train,
    columns=categorical_columns,
    dtype=int
)


# ------------------------------------------------------------
# Handle missing values
# EXACTLY like train_random_forest.py
# ------------------------------------------------------------

X_train = X_train.fillna(0)


print("\nProcessed training matrix:")
print(X_train.shape)


print("\nProcessed training columns:")
for i, col in enumerate(X_train.columns, 1):
    print(f"{i:2}. {col}")


# ------------------------------------------------------------
# Compare with saved model
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MODEL SCHEMA COMPARISON")
print("=" * 70)


print("\nExpected model features:")
for i, col in enumerate(model_features, 1):
    print(f"{i:2}. {col}")


missing_from_processed_train = [
    col
    for col in model_features
    if col not in X_train.columns
]

extra_in_processed_train = [
    col
    for col in X_train.columns
    if col not in model_features
]


print("\nMissing from processed training matrix:")
print(missing_from_processed_train)

print("\nExtra columns in processed training matrix:")
print(extra_in_processed_train)


# ------------------------------------------------------------
# Exact order comparison
# ------------------------------------------------------------

if not missing_from_processed_train and not extra_in_processed_train:

    same_order = (
        list(X_train.columns)
        == list(model_features)
    )

    print("\nFeature order matches model:")
    print(same_order)

    assert same_order, (
        "Feature names exist but column order does not match model."
    )


# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

assert len(X_train.columns) == len(model_features), (
    f"Feature count mismatch: "
    f"training={len(X_train.columns)}, "
    f"model={len(model_features)}"
)

assert list(X_train.columns) == list(model_features), (
    "Final training feature schema does not match model."
)


print("\n" + "=" * 70)
print("✓ MODEL PREPROCESSING VALIDATED")
print("=" * 70)

print(f"\nTraining matrix: {X_train.shape}")
print(f"Model expects:   {len(model_features)} features")
print("Feature names:   MATCH")
print("Feature order:   MATCH")

STEP 23M.10B — MODEL PREPROCESSING VALIDATION

Training feature dataset:
(1017209, 31)


c:\Users\HP\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.9.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\HP\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator RandomForestRegressor from version 1.9.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(



Model type:
<class 'sklearn.ensemble._forest.RandomForestRegressor'>

Model feature count:
26

Missing base features:
[]

Processed training matrix:
(1017209, 26)

Processed training columns:
 1. Store
 2. DayOfWeek
 3. Open
 4. Promo
 5. SchoolHoliday
 6. CompetitionDistance
 7. Year
 8. Month
 9. Day
10. WeekOfYear
11. HasCompetitionDistance
12. HasPromo2
13. StoreAge
14. Sales_Lag_1
15. Sales_Lag_7
16. Sales_Lag_14
17. Sales_Rolling_Mean_7
18. Sales_Rolling_Mean_14
19. Sales_Rolling_Mean_30
20. StoreType_a
21. StoreType_b
22. StoreType_c
23. StoreType_d
24. Assortment_a
25. Assortment_b
26. Assortment_c

MODEL SCHEMA COMPARISON

Expected model features:
 1. Store
 2. DayOfWeek
 3. Open
 4. Promo
 5. SchoolHoliday
 6. CompetitionDistance
 7. Year
 8. Month
 9. Day
10. WeekOfYear
11. HasCompetitionDistance
12. HasPromo2
13. StoreAge
14. Sales_Lag_1
15. Sales_Lag_7
16. Sales_Lag_14
17. Sales_Rolling_Mean_7
18. Sales_Rolling_Mean_14
19. Sales_Rolling_Mean_30
20. StoreType_a
21. StoreTy

##### Step 23M.10C — Preprocess Test Matrix Exactly Like Training

In [103]:
# ============================================================
# Step 23M.10C — Preprocess Test Matrix Exactly Like Training
# ============================================================

print("=" * 70)
print("STEP 23M.10C — TEST MODEL PREPROCESSING")
print("=" * 70)


# ------------------------------------------------------------
# Base model features
# ------------------------------------------------------------

base_features = [
    "Store",
    "DayOfWeek",
    "Open",
    "Promo",
    "SchoolHoliday",
    "CompetitionDistance",
    "StoreType",
    "Assortment",
    "Year",
    "Month",
    "Day",
    "WeekOfYear",
    "HasCompetitionDistance",
    "HasPromo2",
    "StoreAge",
    "Sales_Lag_1",
    "Sales_Lag_7",
    "Sales_Lag_14",
    "Sales_Rolling_Mean_7",
    "Sales_Rolling_Mean_14",
    "Sales_Rolling_Mean_30",
]


# ------------------------------------------------------------
# Check base features exist
# ------------------------------------------------------------

missing_base = [
    col
    for col in base_features
    if col not in test_day.columns
]

print("\nMissing base features:")
print(missing_base)

assert not missing_base, (
    f"Test dataset is missing base features: {missing_base}"
)


# ------------------------------------------------------------
# Create test matrix
# ------------------------------------------------------------

X_test = test_day[base_features].copy()

print("\nRaw test model matrix:")
print(X_test.shape)


# ------------------------------------------------------------
# One-hot encode categorical features
# SAME AS TRAINING
# ------------------------------------------------------------

categorical_columns = [
    "StoreType",
    "Assortment",
]

X_test = pd.get_dummies(
    X_test,
    columns=categorical_columns,
    dtype=int
)


print("\nAfter one-hot encoding:")
print(X_test.shape)

print("\nColumns after one-hot encoding:")
for i, col in enumerate(X_test.columns, 1):
    print(f"{i:2}. {col}")


# ------------------------------------------------------------
# Handle missing values
# SAME AS TRAINING
# ------------------------------------------------------------

missing_before = X_test.isna().sum()

print("\nMissing values before fillna:")
print(
    missing_before[missing_before > 0]
)


X_test = X_test.fillna(0)


print("\nMissing values after fillna:")
print(
    X_test.isna().sum().sum()
)


# ------------------------------------------------------------
# Align EXACTLY with saved model
# ------------------------------------------------------------

X_test = X_test.reindex(
    columns=model_features,
    fill_value=0
)


# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL TEST MATRIX")
print("=" * 70)

print("\nShape:")
print(X_test.shape)

print("\nExpected:")
print((len(test_day), len(model_features)))


print("\nFinal columns:")

for i, col in enumerate(X_test.columns, 1):
    print(f"{i:2}. {col}")


# ------------------------------------------------------------
# Check feature names
# ------------------------------------------------------------

assert list(X_test.columns) == list(model_features), (
    "Test feature order does not match model."
)


# ------------------------------------------------------------
# Check missing values
# ------------------------------------------------------------

missing_after = X_test.isna().sum()

print("\nMissing values:")
print(
    missing_after[missing_after > 0]
)

assert X_test.isna().sum().sum() == 0, (
    "Test matrix still contains missing values."
)


# ------------------------------------------------------------
# Check row count
# ------------------------------------------------------------

assert len(X_test) == len(test_day), (
    "Test row count changed during preprocessing."
)


# ------------------------------------------------------------
# Check feature count
# ------------------------------------------------------------

assert X_test.shape[1] == len(model_features), (
    f"Expected {len(model_features)} features, "
    f"got {X_test.shape[1]}"
)


print("\n" + "=" * 70)
print("✓ STEP 23M.10C PASSED")
print("=" * 70)

print(f"\nTest rows:     {X_test.shape[0]}")
print(f"Test features: {X_test.shape[1]}")
print(f"Model expects: {len(model_features)}")

print("\n✓ Feature names match")
print("✓ Feature order matches")
print("✓ No missing values")
print("✓ Row count preserved")
print("✓ Test matrix ready for prediction")

STEP 23M.10C — TEST MODEL PREPROCESSING

Missing base features:
[]

Raw test model matrix:
(856, 21)

After one-hot encoding:
(856, 26)

Columns after one-hot encoding:
 1. Store
 2. DayOfWeek
 3. Open
 4. Promo
 5. SchoolHoliday
 6. CompetitionDistance
 7. Year
 8. Month
 9. Day
10. WeekOfYear
11. HasCompetitionDistance
12. HasPromo2
13. StoreAge
14. Sales_Lag_1
15. Sales_Lag_7
16. Sales_Lag_14
17. Sales_Rolling_Mean_7
18. Sales_Rolling_Mean_14
19. Sales_Rolling_Mean_30
20. StoreType_a
21. StoreType_b
22. StoreType_c
23. StoreType_d
24. Assortment_a
25. Assortment_b
26. Assortment_c

Missing values before fillna:
CompetitionDistance      2
StoreAge               317
dtype: int64

Missing values after fillna:
0

FINAL TEST MATRIX

Shape:
(856, 26)

Expected:
(856, 26)

Final columns:
 1. Store
 2. DayOfWeek
 3. Open
 4. Promo
 5. SchoolHoliday
 6. CompetitionDistance
 7. Year
 8. Month
 9. Day
10. WeekOfYear
11. HasCompetitionDistance
12. HasPromo2
13. StoreAge
14. Sales_Lag_1
15. Sale

##### Step 23M.11 — Run predictions

In [104]:
print("=" * 70)
print("STEP 23M.11 — GENERATING TEST PREDICTIONS")
print("=" * 70)


# ------------------------------------------------------------
# Predict
# ------------------------------------------------------------

predictions = model.predict(X_test)


# ------------------------------------------------------------
# Basic validation
# ------------------------------------------------------------

print("\nPrediction type:")
print(type(predictions))

print("\nPrediction shape:")
print(predictions.shape)

print("\nExpected rows:")
print(len(test_day))


# ------------------------------------------------------------
# Check for invalid values
# ------------------------------------------------------------

print("\nMissing predictions:")
print(pd.isna(predictions).sum())

print("\nInfinite predictions:")
print(~pd.Series(predictions).apply(pd.api.types.is_number).all())


# ------------------------------------------------------------
# Prediction statistics
# ------------------------------------------------------------

prediction_series = pd.Series(predictions)

print("\n" + "=" * 70)
print("PREDICTION STATISTICS")
print("=" * 70)

print(
    prediction_series.describe()
)


# ------------------------------------------------------------
# First 20 predictions
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FIRST 20 PREDICTIONS")
print("=" * 70)

prediction_preview = test_day[
    ["Id", "Store", "Date", "Open", "Promo"]
].copy()

prediction_preview["Prediction"] = predictions

print(
    prediction_preview.head(20).to_string(index=False)
)


# ------------------------------------------------------------
# Check negative predictions
# ------------------------------------------------------------

negative_count = (predictions < 0).sum()

print("\nNegative predictions:")
print(negative_count)


# ------------------------------------------------------------
# Check zero predictions
# ------------------------------------------------------------

zero_count = (predictions == 0).sum()

print("\nZero predictions:")
print(zero_count)


# ------------------------------------------------------------
# Final assertions
# ------------------------------------------------------------

assert len(predictions) == len(test_day), (
    "Prediction row count does not match test rows."
)

assert not pd.isna(predictions).any(), (
    "Predictions contain NaN values."
)

assert pd.Series(predictions).notna().all(), (
    "Predictions contain invalid missing values."
)


print("\n" + "=" * 70)
print("✓ STEP 23M.11 PASSED")
print("=" * 70)

print(f"\nPredictions generated: {len(predictions)}")
print("✓ Row count matches test data")
print("✓ No NaN predictions")
print("✓ Prediction generation successful")

STEP 23M.11 — GENERATING TEST PREDICTIONS

Prediction type:
<class 'numpy.ndarray'>

Prediction shape:
(856,)

Expected rows:
856

Missing predictions:
0

Infinite predictions:
False

PREDICTION STATISTICS
count      856.000000
mean      6534.869035
std       2722.910946
min       1220.715819
25%       4724.746379
50%       6200.228579
75%       7990.247501
max      24772.135532
dtype: float64

FIRST 20 PREDICTIONS
   Id  Store       Date  Open  Promo  Prediction
40233      1 2015-08-01   1.0      0 4986.414667
40234      3 2015-08-01   1.0      0 4491.336266
40235      7 2015-08-01   1.0      0 7590.291401
40236      8 2015-08-01   1.0      0 4032.802249
40237      9 2015-08-01   1.0      0 6381.006990
40238     10 2015-08-01   1.0      0 6244.917801
40239     11 2015-08-01   1.0      0 5973.893580
40240     12 2015-08-01   1.0      0 6992.672405
40241     13 2015-08-01   1.0      0 5234.403937
40242     14 2015-08-01   1.0      0 3534.955263
40243     15 2015-08-01   1.0      0 6196.

In [107]:
print("\nMissing predictions:")
print(pd.isna(predictions).sum())

print("\nInfinite predictions:")
print(np.isinf(predictions).sum())


Missing predictions:
0

Infinite predictions:
0


##### Step 23M.12 — Build the submission dataframe

In [108]:
print("=" * 70)
print("STEP 23M.12 — BUILDING SUBMISSION DATAFRAME")
print("=" * 70)


# ------------------------------------------------------------
# Create submission
# ------------------------------------------------------------

submission = pd.DataFrame({
    "Id": test_day["Id"].to_numpy(),
    "Sales": predictions
})


# ------------------------------------------------------------
# Validate shape
# ------------------------------------------------------------

print("\nSubmission shape:")
print(submission.shape)

print("\nExpected:")
print((len(test_day), 2))


# ------------------------------------------------------------
# Validate columns
# ------------------------------------------------------------

print("\nSubmission columns:")
print(list(submission.columns))


assert list(submission.columns) == [
    "Id",
    "Sales"
], (
    "Submission columns do not match expected schema."
)


# ------------------------------------------------------------
# Validate row count
# ------------------------------------------------------------

assert len(submission) == len(test_day), (
    "Submission row count does not match test data."
)


# ------------------------------------------------------------
# Validate IDs
# ------------------------------------------------------------

print("\nID statistics:")
print(submission["Id"].describe())

print("\nUnique IDs:")
print(submission["Id"].nunique())

print("\nExpected unique IDs:")
print(test_day["Id"].nunique())


assert submission["Id"].nunique() == len(submission), (
    "Duplicate Id values found in submission."
)


# ------------------------------------------------------------
# Validate Sales
# ------------------------------------------------------------

print("\nSales missing:")
print(submission["Sales"].isna().sum())

print("\nSales infinite:")
print(np.isinf(submission["Sales"]).sum())

print("\nSales negative:")
print((submission["Sales"] < 0).sum())


assert submission["Sales"].notna().all(), (
    "Submission contains missing Sales values."
)

assert np.isfinite(submission["Sales"]).all(), (
    "Submission contains infinite Sales values."
)

assert (submission["Sales"] >= 0).all(), (
    "Submission contains negative Sales values."
)


# ------------------------------------------------------------
# Preview
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SUBMISSION PREVIEW")
print("=" * 70)

print("\nFirst 10:")
print(
    submission.head(10).to_string(index=False)
)

print("\nLast 10:")
print(
    submission.tail(10).to_string(index=False)
)


# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("✓ STEP 23M.12 PASSED")
print("=" * 70)

print(f"\nRows: {len(submission)}")
print(f"Columns: {list(submission.columns)}")
print("✓ IDs unique")
print("✓ Sales contains no NaN")
print("✓ Sales contains no infinity")
print("✓ Sales contains no negative values")
print("✓ Submission schema is correct")

STEP 23M.12 — BUILDING SUBMISSION DATAFRAME

Submission shape:
(856, 2)

Expected:
(856, 2)

Submission columns:
['Id', 'Sales']

ID statistics:
count      856.000000
mean     40660.500000
std        247.250211
min      40233.000000
25%      40446.750000
50%      40660.500000
75%      40874.250000
max      41088.000000
Name: Id, dtype: float64

Unique IDs:
856

Expected unique IDs:
856

Sales missing:
0

Sales infinite:
0

Sales negative:
0

SUBMISSION PREVIEW

First 10:
   Id       Sales
40233 4986.414667
40234 4491.336266
40235 7590.291401
40236 4032.802249
40237 6381.006990
40238 6244.917801
40239 5973.893580
40240 6992.672405
40241 5234.403937
40242 3534.955263

Last 10:
   Id        Sales
41079  6291.327510
41080  4788.983900
41081  3391.266305
41082  6412.960825
41083  3950.422242
41084  2744.041770
41085  7463.406191
41086  5953.793072
41087 24772.135532
41088  7946.851410

✓ STEP 23M.12 PASSED

Rows: 856
Columns: ['Id', 'Sales']
✓ IDs unique
✓ Sales contains no NaN
✓ Sales cont

##### Step 23M.13 — Build complete 41,088-row test feature matrix

In [111]:
# ============================================================
# STEP 23M.13 — VALIDATE COMPLETE TEST DATA
# ============================================================

print("\n" + "=" * 70)
print("COMPLETE TEST DATA VALIDATION")
print("=" * 70)

print("\nTest shape:")
print(test_full.shape)

print("\nDate range:")
print(
    test_full["Date"].min(),
    "→",
    test_full["Date"].max()
)

print("\nUnique stores:")
print(test_full["Store"].nunique())

print("\nExpected row count:")
print(41088)

# ------------------------------------------------------------
# Required assertions
# ------------------------------------------------------------

assert len(test_full) == 41088, (
    f"Expected 41088 rows, got {len(test_full)}"
)

assert test_full["Id"].is_unique, (
    "Test Id values are not unique"
)

assert test_full["Date"].notna().all(), (
    "Missing dates found"
)

# Store IDs do NOT need to be all 1115.
# Some stores have no test rows / are closed on particular dates.

print("\nStore metadata columns available:")

for col in required_store_columns:
    print(
        f"{col}: "
        f"{col in test_full.columns}"
    )

print("\n✓ Complete test dataset validated")
print(f"✓ Rows: {len(test_full)}")
print(f"✓ Unique test stores: {test_full['Store'].nunique()}")
print("✓ Store metadata merged")
print("✓ Test IDs unique")


COMPLETE TEST DATA VALIDATION

Test shape:
(41088, 17)

Date range:
2015-08-01 00:00:00 → 2015-09-17 00:00:00

Unique stores:
856

Expected row count:
41088

Store metadata columns available:
CompetitionDistance: True
CompetitionOpenSinceMonth: True
CompetitionOpenSinceYear: True
Promo2: True
Promo2SinceWeek: True
Promo2SinceYear: True
PromoInterval: True
StoreType: True
Assortment: True

✓ Complete test dataset validated
✓ Rows: 41088
✓ Unique test stores: 856
✓ Store metadata merged
✓ Test IDs unique


In [112]:
print(
    test_full[
        [
            "Id",
            "Store",
            "Date",
            "CompetitionDistance",
            "CompetitionOpenSinceYear",
            "Promo2",
            "StoreType",
            "Assortment"
        ]
    ].head(10)
)

   Id  Store       Date  CompetitionDistance  CompetitionOpenSinceYear  \
0   1      1 2015-09-17               1270.0                    2008.0   
1   2      3 2015-09-17              14130.0                    2006.0   
2   3      7 2015-09-17              24000.0                    2013.0   
3   4      8 2015-09-17               7520.0                    2014.0   
4   5      9 2015-09-17               2030.0                    2000.0   
5   6     10 2015-09-17               3160.0                    2009.0   
6   7     11 2015-09-17                960.0                    2011.0   
7   8     12 2015-09-17               1070.0                       NaN   
8   9     13 2015-09-17                310.0                       NaN   
9  10     14 2015-09-17               1300.0                    2014.0   

   Promo2 StoreType Assortment  
0       0         c          a  
1       1         a          a  
2       0         a          c  
3       0         a          a  
4       0         a 

##### Step 23M.14: generate the historical/lag/rolling features for all 41,088 test rows.

In [113]:
# ============================================================
# STEP 23M.14 — BUILD HISTORICAL FEATURES FOR COMPLETE TEST
# ============================================================

print("\n" + "=" * 70)
print("STEP 23M.14 — COMPLETE TEST HISTORICAL FEATURES")
print("=" * 70)

# ------------------------------------------------------------
# Load historical sales
# ------------------------------------------------------------

historical_source = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "train_features.csv",
    low_memory=False,
    dtype={"StateHoliday": "string"}
)

historical_source["Date"] = pd.to_datetime(
    historical_source["Date"]
)

print("\nHistorical source:")
print(historical_source.shape)

print(
    "Historical date range:",
    historical_source["Date"].min(),
    "→",
    historical_source["Date"].max()
)

# ------------------------------------------------------------
# Keep only columns required for historical calculations
# ------------------------------------------------------------

sales_history = historical_source[
    [
        "Store",
        "Date",
        "Sales"
    ]
].copy()

sales_history = sales_history.sort_values(
    ["Store", "Date"]
)

print("\nSales history shape:")
print(sales_history.shape)

# ------------------------------------------------------------
# Create lookup indexed by Store + Date
# ------------------------------------------------------------

sales_lookup = (
    sales_history
    .set_index(["Store", "Date"])["Sales"]
)

# ------------------------------------------------------------
# Function to calculate historical features
# ------------------------------------------------------------

def get_historical_features(
    store_id,
    current_date,
    sales_history
):

    store_history = sales_history[
        sales_history["Store"] == store_id
    ].copy()

    store_history = store_history[
        store_history["Date"] < current_date
    ].sort_values("Date")

    if len(store_history) < 30:
        raise ValueError(
            f"Insufficient history for Store {store_id} "
            f"before {current_date}"
        )

    sales = store_history["Sales"]

    return {
        "Sales_Lag_1": sales.iloc[-1],
        "Sales_Lag_7": sales.iloc[-7],
        "Sales_Lag_14": sales.iloc[-14],

        "Sales_Rolling_Mean_7":
            sales.iloc[-7:].mean(),

        "Sales_Rolling_Mean_14":
            sales.iloc[-14:].mean(),

        "Sales_Rolling_Mean_30":
            sales.iloc[-30:].mean(),
    }


# ------------------------------------------------------------
# Generate historical features
# ------------------------------------------------------------

historical_rows = []

for idx, row in test_full.iterrows():

    historical = get_historical_features(
        store_id=int(row["Store"]),
        current_date=row["Date"],
        sales_history=sales_history
    )

    historical_rows.append({
        "Id": row["Id"],
        "Store": row["Store"],
        "Date": row["Date"],
        **historical
    })


test_historical = pd.DataFrame(
    historical_rows
)

# ------------------------------------------------------------
# Validate
# ------------------------------------------------------------

print("\nHistorical feature shape:")
print(test_historical.shape)

print("\nHistorical feature columns:")
print(test_historical.columns.tolist())

historical_feature_columns = [
    "Sales_Lag_1",
    "Sales_Lag_7",
    "Sales_Lag_14",
    "Sales_Rolling_Mean_7",
    "Sales_Rolling_Mean_14",
    "Sales_Rolling_Mean_30",
]

print("\nMissing historical values:")

print(
    test_historical[
        historical_feature_columns
    ].isna().sum()
)

assert len(test_historical) == 41088

assert (
    test_historical["Id"].nunique()
    == 41088
)

assert (
    test_historical[
        historical_feature_columns
    ].notna().all().all()
)

print("\n" + "=" * 70)
print("✓ STEP 23M.14 PASSED")
print("=" * 70)

print(f"Rows: {len(test_historical)}")
print("✓ Lag features generated")
print("✓ Rolling features generated")
print("✓ No missing historical features")
print("✓ One historical feature row per test ID")


STEP 23M.14 — COMPLETE TEST HISTORICAL FEATURES

Historical source:
(1017209, 31)
Historical date range: 2013-01-01 00:00:00 → 2015-07-31 00:00:00

Sales history shape:
(1017209, 3)

Historical feature shape:
(41088, 9)

Historical feature columns:
['Id', 'Store', 'Date', 'Sales_Lag_1', 'Sales_Lag_7', 'Sales_Lag_14', 'Sales_Rolling_Mean_7', 'Sales_Rolling_Mean_14', 'Sales_Rolling_Mean_30']

Missing historical values:
Sales_Lag_1              0
Sales_Lag_7              0
Sales_Lag_14             0
Sales_Rolling_Mean_7     0
Sales_Rolling_Mean_14    0
Sales_Rolling_Mean_30    0
dtype: int64

✓ STEP 23M.14 PASSED
Rows: 41088
✓ Lag features generated
✓ Rolling features generated
✓ No missing historical features
✓ One historical feature row per test ID


##### STEP 23M.15 — MERGE COMPLETE TEST WITH HISTORICAL FEATURES

In [114]:
# ============================================================
# STEP 23M.15 — MERGE COMPLETE TEST WITH HISTORICAL FEATURES
# ============================================================

print("\n" + "=" * 70)
print("STEP 23M.15 — MERGE TEST + HISTORICAL FEATURES")
print("=" * 70)

# ------------------------------------------------------------
# Historical feature columns
# ------------------------------------------------------------

historical_feature_columns = [
    "Sales_Lag_1",
    "Sales_Lag_7",
    "Sales_Lag_14",
    "Sales_Rolling_Mean_7",
    "Sales_Rolling_Mean_14",
    "Sales_Rolling_Mean_30",
]

# ------------------------------------------------------------
# Check test_full before merge
# ------------------------------------------------------------

print("\nComplete test before merge:")
print(test_full.shape)

print("Unique IDs:", test_full["Id"].nunique())
print("Unique stores:", test_full["Store"].nunique())

# ------------------------------------------------------------
# Check historical table
# ------------------------------------------------------------

print("\nHistorical feature table:")
print(test_historical.shape)

print("Unique IDs:", test_historical["Id"].nunique())
print("Unique stores:", test_historical["Store"].nunique())

assert len(test_full) == 41088
assert test_full["Id"].nunique() == 41088

assert len(test_historical) == 41088
assert test_historical["Id"].nunique() == 41088

# ------------------------------------------------------------
# Prevent accidental duplicate columns
# ------------------------------------------------------------

test_base = test_full.drop(
    columns=[
        col
        for col in historical_feature_columns
        if col in test_full.columns
    ],
    errors="ignore"
).copy()

# ------------------------------------------------------------
# Merge by Id
# ------------------------------------------------------------

test_complete = test_base.merge(
    test_historical[
        ["Id"] + historical_feature_columns
    ],
    on="Id",
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# Validate merge
# ------------------------------------------------------------

print("\nMerged test shape:")
print(test_complete.shape)

print("\nUnique IDs after merge:")
print(test_complete["Id"].nunique())

print("\nMissing historical features after merge:")

print(
    test_complete[
        historical_feature_columns
    ].isna().sum()
)

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert len(test_complete) == 41088

assert test_complete["Id"].nunique() == 41088

assert test_complete["Store"].nunique() == 856

assert test_complete[
    historical_feature_columns
].notna().all().all()

# Verify ID preservation
assert set(test_complete["Id"]) == set(test_full["Id"])

# ------------------------------------------------------------
# Check duplicate IDs
# ------------------------------------------------------------

duplicate_ids = (
    test_complete["Id"]
    .duplicated()
    .sum()
)

print("\nDuplicate IDs:")
print(duplicate_ids)

assert duplicate_ids == 0

# ------------------------------------------------------------
# Preview
# ------------------------------------------------------------

print("\nFirst 10 merged rows:")

print(
    test_complete[
        [
            "Id",
            "Store",
            "Date",
            "Sales_Lag_1",
            "Sales_Lag_7",
            "Sales_Lag_14",
            "Sales_Rolling_Mean_7",
            "Sales_Rolling_Mean_14",
            "Sales_Rolling_Mean_30",
        ]
    ].head(10)
)

print("\n" + "=" * 70)
print("✓ STEP 23M.15 PASSED")
print("=" * 70)

print("Rows:", len(test_complete))
print("Unique IDs:", test_complete["Id"].nunique())
print("Unique stores:", test_complete["Store"].nunique())

print("✓ One-to-one merge")
print("✓ All test IDs preserved")
print("✓ No duplicate IDs")
print("✓ All historical features present")


STEP 23M.15 — MERGE TEST + HISTORICAL FEATURES

Complete test before merge:
(41088, 17)
Unique IDs: 41088
Unique stores: 856

Historical feature table:
(41088, 9)
Unique IDs: 41088
Unique stores: 856

Merged test shape:
(41088, 23)

Unique IDs after merge:
41088

Missing historical features after merge:
Sales_Lag_1              0
Sales_Lag_7              0
Sales_Lag_14             0
Sales_Rolling_Mean_7     0
Sales_Rolling_Mean_14    0
Sales_Rolling_Mean_30    0
dtype: int64

Duplicate IDs:
0

First 10 merged rows:
   Id  Store       Date  Sales_Lag_1  Sales_Lag_7  Sales_Lag_14  \
0   1      1 2015-09-17         5263         4364          4406   
1   2      3 2015-09-17         8314         3878          4258   
2   3      7 2015-09-17        15344         6574          6861   
3   4      8 2015-09-17         8492         3288          3813   
4   5      9 2015-09-17         8565         5330          5987   
5   6     10 2015-09-17         7185         4713          5761   
6   7    

##### STEP 23M.16 — BUILD COMPLETE RAW MODEL FEATURE MATRIX

In [116]:
# ============================================================
# STEP 23M.16 — BUILD COMPLETE RAW MODEL FEATURE MATRIX
# ============================================================

print("\n" + "=" * 70)
print("STEP 23M.16 — COMPLETE RAW MODEL FEATURE MATRIX")
print("=" * 70)

# ------------------------------------------------------------
# Model raw feature schema
# ------------------------------------------------------------

model_raw_features = [
    "Store",
    "DayOfWeek",
    "Open",
    "Promo",
    "SchoolHoliday",
    "CompetitionDistance",
    "StoreType",
    "Assortment",
    "Year",
    "Month",
    "Day",
    "WeekOfYear",
    "HasCompetitionDistance",
    "HasPromo2",
    "StoreAge",
    "Sales_Lag_1",
    "Sales_Lag_7",
    "Sales_Lag_14",
    "Sales_Rolling_Mean_7",
    "Sales_Rolling_Mean_14",
    "Sales_Rolling_Mean_30",
]

assert len(model_raw_features) == 21

# ------------------------------------------------------------
# Make sure Date is datetime
# ------------------------------------------------------------

test_complete["Date"] = pd.to_datetime(
    test_complete["Date"],
    errors="coerce"
)

assert test_complete["Date"].notna().all()

# ------------------------------------------------------------
# Date features
# ------------------------------------------------------------

test_complete["Year"] = test_complete["Date"].dt.year
test_complete["Month"] = test_complete["Date"].dt.month
test_complete["Day"] = test_complete["Date"].dt.day

test_complete["WeekOfYear"] = (
    test_complete["Date"]
    .dt.isocalendar()
    .week
    .astype(int)
)

# ------------------------------------------------------------
# Business features
# ------------------------------------------------------------

test_complete["HasCompetitionDistance"] = (
    test_complete["CompetitionDistance"]
    .notna()
    .astype(int)
)

test_complete["HasPromo2"] = (
    test_complete["Promo2"]
    .fillna(0)
    .astype(int)
)

test_complete["StoreAge"] = (
    test_complete["Year"]
    - test_complete["CompetitionOpenSinceYear"]
)

test_complete["StoreAge"] = (
    test_complete["StoreAge"]
    .clip(lower=0)
)

# ------------------------------------------------------------
# Check all required features exist
# ------------------------------------------------------------

missing_raw_features = [
    col
    for col in model_raw_features
    if col not in test_complete.columns
]

print("\nMissing raw model features:")
print(missing_raw_features)

assert not missing_raw_features

# ------------------------------------------------------------
# Build raw model matrix
# ------------------------------------------------------------

X_test_full_raw = test_complete[
    model_raw_features
].copy()

print("\nRaw complete test matrix:")
print(X_test_full_raw.shape)

assert X_test_full_raw.shape == (41088, 21)

# ------------------------------------------------------------
# Validate categorical values
# ------------------------------------------------------------

print("\nStoreType values:")
print(
    sorted(
        X_test_full_raw["StoreType"]
        .dropna()
        .unique()
        .tolist()
    )
)

print("\nAssortment values:")
print(
    sorted(
        X_test_full_raw["Assortment"]
        .dropna()
        .unique()
        .tolist()
    )
)

assert set(
    X_test_full_raw["StoreType"]
    .dropna()
    .unique()
).issubset({"a", "b", "c", "d"})

assert set(
    X_test_full_raw["Assortment"]
    .dropna()
    .unique()
).issubset({"a", "b", "c"})

# ------------------------------------------------------------
# Missing-value inspection
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RAW MODEL MATRIX MISSING VALUES")
print("=" * 70)

raw_missing = X_test_full_raw.isna().sum()

print(
    raw_missing[
        raw_missing > 0
    ]
)

# ------------------------------------------------------------
# Show all missing-value counts explicitly
# ------------------------------------------------------------

print("\nCompetitionDistance missing:")
print(
    X_test_full_raw[
        "CompetitionDistance"
    ].isna().sum()
)

print("\nStoreAge missing:")
print(
    X_test_full_raw[
        "StoreAge"
    ].isna().sum()
)

# ------------------------------------------------------------
# Historical features MUST have no missing values
# ------------------------------------------------------------

historical_columns = [
    "Sales_Lag_1",
    "Sales_Lag_7",
    "Sales_Lag_14",
    "Sales_Rolling_Mean_7",
    "Sales_Rolling_Mean_14",
    "Sales_Rolling_Mean_30",
]

historical_missing = (
    X_test_full_raw[
        historical_columns
    ]
    .isna()
    .sum()
)

print("\nHistorical feature missing values:")
print(historical_missing)

assert historical_missing.sum() == 0

# ------------------------------------------------------------
# Validate row count and ID preservation
# ------------------------------------------------------------

assert len(X_test_full_raw) == 41088

assert test_complete["Id"].nunique() == 41088

# ------------------------------------------------------------
# Verify test date range
# ------------------------------------------------------------

print("\nDate range:")
print(
    X_test_full_raw.index.min(),
    "to",
    X_test_full_raw.index.max()
)

print(
    "\nActual test dates:",
    test_complete["Date"].min(),
    "→",
    test_complete["Date"].max()
)

assert (
    test_complete["Date"].min()
    == pd.Timestamp("2015-08-01")
)

assert (
    test_complete["Date"].max()
    == pd.Timestamp("2015-09-17")
)

# ------------------------------------------------------------
# Preview
# ------------------------------------------------------------

print("\nFirst 10 rows:")

print(
    X_test_full_raw.head(10)
)

print("\n" + "=" * 70)
print("✓ STEP 23M.16 PASSED")
print("=" * 70)

print("Rows:", len(X_test_full_raw))
print("Raw features:", X_test_full_raw.shape[1])

print("✓ 41,088 rows preserved")
print("✓ Exactly 21 raw model features")
print("✓ Date features created")
print("✓ Business features created")
print("✓ Historical features present")
print("✓ Historical features have zero missing values")
print("✓ Raw matrix ready for preprocessing")


STEP 23M.16 — COMPLETE RAW MODEL FEATURE MATRIX

Missing raw model features:
[]

Raw complete test matrix:
(41088, 21)

StoreType values:
['a', 'b', 'c', 'd']

Assortment values:
['a', 'b', 'c']

RAW MODEL MATRIX MISSING VALUES
Open                      11
CompetitionDistance       96
StoreAge               15216
dtype: int64

CompetitionDistance missing:
96

StoreAge missing:
15216

Historical feature missing values:
Sales_Lag_1              0
Sales_Lag_7              0
Sales_Lag_14             0
Sales_Rolling_Mean_7     0
Sales_Rolling_Mean_14    0
Sales_Rolling_Mean_30    0
dtype: int64

Date range:
0 to 41087

Actual test dates: 2015-08-01 00:00:00 → 2015-09-17 00:00:00

First 10 rows:
   Store  DayOfWeek  Open  Promo  SchoolHoliday  CompetitionDistance  \
0      1          4   1.0      1              0               1270.0   
1      3          4   1.0      1              0              14130.0   
2      7          4   1.0      1              0              24000.0   
3      8    

##### STEP 23M.17 — COMPLETE TEST MODEL PREPROCESSING

In [117]:
print("\n" + "=" * 70)
print("STEP 23M.17 — COMPLETE TEST MODEL PREPROCESSING")
print("=" * 70)

# ------------------------------------------------------------
# Load model package
# ------------------------------------------------------------

model_package = joblib.load(
    PROJECT_ROOT
    / "models"
    / "random_forest_final.pkl"
)

model = model_package["model"]
model_features = model_package["features"]

print("\nModel expected feature count:")
print(len(model_features))

print("\nModel feature schema:")
for i, col in enumerate(model_features, 1):
    print(f"{i:2}. {col}")

assert len(model_features) == 26
assert model.n_features_in_ == 26

# ------------------------------------------------------------
# Copy raw matrix
# ------------------------------------------------------------

X_test_full = X_test_full_raw.copy()

print("\nRaw test matrix:")
print(X_test_full.shape)

assert X_test_full.shape == (41088, 21)

# ------------------------------------------------------------
# One-hot encoding
# ------------------------------------------------------------

categorical_columns = [
    "StoreType",
    "Assortment",
]

X_test_full = pd.get_dummies(
    X_test_full,
    columns=categorical_columns,
    dtype=int
)

print("\nAfter one-hot encoding:")
print(X_test_full.shape)

print("\nColumns after one-hot encoding:")

for i, col in enumerate(X_test_full.columns, 1):
    print(f"{i:2}. {col}")

# ------------------------------------------------------------
# Check all model columns exist
# ------------------------------------------------------------

missing_model_features = [
    col
    for col in model_features
    if col not in X_test_full.columns
]

extra_features = [
    col
    for col in X_test_full.columns
    if col not in model_features
]

print("\nMissing model features:")
print(missing_model_features)

print("\nExtra features:")
print(extra_features)

assert not missing_model_features
assert not extra_features

# ------------------------------------------------------------
# Enforce exact model feature order
# ------------------------------------------------------------

X_test_full = X_test_full[
    model_features
].copy()

print("\nAfter enforcing model feature order:")
print(X_test_full.shape)

assert X_test_full.shape == (41088, 26)

# ------------------------------------------------------------
# Missing values before fillna
# ------------------------------------------------------------

print("\nMissing values before fillna:")

missing_before_fill = (
    X_test_full.isna().sum()
)

print(
    missing_before_fill[
        missing_before_fill > 0
    ]
)

# ------------------------------------------------------------
# Training-compatible missing-value handling
# ------------------------------------------------------------

X_test_full = X_test_full.fillna(0)

# ------------------------------------------------------------
# Missing values after fillna
# ------------------------------------------------------------

missing_after_fill = (
    X_test_full.isna().sum().sum()
)

print("\nTotal missing values after fillna:")
print(missing_after_fill)

assert missing_after_fill == 0

# ------------------------------------------------------------
# Final schema comparison
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL MODEL SCHEMA COMPARISON")
print("=" * 70)

print("\nExpected model columns:")
for i, col in enumerate(model_features, 1):
    print(f"{i:2}. {col}")

print("\nFinal test columns:")
for i, col in enumerate(X_test_full.columns, 1):
    print(f"{i:2}. {col}")

# ------------------------------------------------------------
# Exact feature-name and order validation
# ------------------------------------------------------------

feature_names_match = (
    list(X_test_full.columns)
    == list(model_features)
)

print("\nFeature names and order match:")
print(feature_names_match)

assert feature_names_match

# ------------------------------------------------------------
# Final dimensions
# ------------------------------------------------------------

print("\nFinal complete test matrix:")
print(X_test_full.shape)

assert X_test_full.shape == (
    41088,
    26
)

# ------------------------------------------------------------
# Row preservation
# ------------------------------------------------------------

assert len(X_test_full) == 41088

# ------------------------------------------------------------
# Numeric validation
# ------------------------------------------------------------

non_numeric_columns = (
    X_test_full
    .select_dtypes(exclude="number")
    .columns
    .tolist()
)

print("\nNon-numeric columns:")
print(non_numeric_columns)

assert not non_numeric_columns

# ------------------------------------------------------------
# Final missing-value validation
# ------------------------------------------------------------

print("\nFinal missing values:")
print(
    X_test_full.isna().sum().sum()
)

assert X_test_full.isna().sum().sum() == 0

# ------------------------------------------------------------
# Preview
# ------------------------------------------------------------

print("\nFirst 10 processed rows:")

print(
    X_test_full.head(10)
)

print("\n" + "=" * 70)
print("✓ STEP 23M.17 PASSED")
print("=" * 70)

print("Rows:", len(X_test_full))
print("Features:", X_test_full.shape[1])
print("Model expects:", model.n_features_in_)

print("✓ One-hot encoding completed")
print("✓ Missing values filled")
print("✓ Feature names match")
print("✓ Feature order matches")
print("✓ No extra features")
print("✓ No missing values")
print("✓ 41,088 rows preserved")
print("✓ Complete test matrix ready for prediction")


STEP 23M.17 — COMPLETE TEST MODEL PREPROCESSING


c:\Users\HP\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.9.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(



Model expected feature count:
26

Model feature schema:
 1. Store
 2. DayOfWeek
 3. Open
 4. Promo
 5. SchoolHoliday
 6. CompetitionDistance
 7. Year
 8. Month
 9. Day
10. WeekOfYear
11. HasCompetitionDistance
12. HasPromo2
13. StoreAge
14. Sales_Lag_1
15. Sales_Lag_7
16. Sales_Lag_14
17. Sales_Rolling_Mean_7
18. Sales_Rolling_Mean_14
19. Sales_Rolling_Mean_30
20. StoreType_a
21. StoreType_b
22. StoreType_c
23. StoreType_d
24. Assortment_a
25. Assortment_b
26. Assortment_c

Raw test matrix:
(41088, 21)

After one-hot encoding:
(41088, 26)

Columns after one-hot encoding:
 1. Store
 2. DayOfWeek
 3. Open
 4. Promo
 5. SchoolHoliday
 6. CompetitionDistance
 7. Year
 8. Month
 9. Day
10. WeekOfYear
11. HasCompetitionDistance
12. HasPromo2
13. StoreAge
14. Sales_Lag_1
15. Sales_Lag_7
16. Sales_Lag_14
17. Sales_Rolling_Mean_7
18. Sales_Rolling_Mean_14
19. Sales_Rolling_Mean_30
20. StoreType_a
21. StoreType_b
22. StoreType_c
23. StoreType_d
24. Assortment_a
25. Assortment_b
26. Assortment_c

c:\Users\HP\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator RandomForestRegressor from version 1.9.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


##### STEP 23M.18 — GENERATE COMPLETE TEST PREDICTIONS

In [118]:
# ============================================================
# STEP 23M.18 — GENERATE COMPLETE TEST PREDICTIONS
# ============================================================

print("\n" + "=" * 70)
print("STEP 23M.18 — GENERATE PREDICTIONS FOR ALL 41,088 TEST ROWS")
print("=" * 70)

# ------------------------------------------------------------
# Validate inputs before prediction
# ------------------------------------------------------------

print("\nTest feature matrix:")
print(X_test_full.shape)

print("\nModel expected features:")
print(model.n_features_in_)

assert X_test_full.shape == (41088, 26)
assert model.n_features_in_ == 26

assert list(X_test_full.columns) == list(
    model.feature_names_in_
)

assert X_test_full.isna().sum().sum() == 0

print("\n✓ Input matrix validated")
print("✓ Shape: (41088, 26)")
print("✓ Feature names match")
print("✓ Feature order matches")
print("✓ No missing values")

# ------------------------------------------------------------
# Generate predictions
# ------------------------------------------------------------

print("\nGenerating predictions...")

predictions_full = model.predict(
    X_test_full
)

# ------------------------------------------------------------
# Basic prediction validation
# ------------------------------------------------------------

print("\nPrediction type:")
print(type(predictions_full))

print("\nPrediction shape:")
print(predictions_full.shape)

assert isinstance(
    predictions_full,
    np.ndarray
)

assert predictions_full.shape == (41088,)

# ------------------------------------------------------------
# Missing / infinite validation
# ------------------------------------------------------------

missing_predictions = int(
    np.isnan(predictions_full).sum()
)

infinite_predictions = int(
    np.isinf(predictions_full).sum()
)

negative_predictions = int(
    (predictions_full < 0).sum()
)

zero_predictions = int(
    (predictions_full == 0).sum()
)

print("\nPrediction validation:")
print(
    "Missing predictions:",
    missing_predictions
)

print(
    "Infinite predictions:",
    infinite_predictions
)

print(
    "Negative predictions:",
    negative_predictions
)

print(
    "Zero predictions:",
    zero_predictions
)

assert missing_predictions == 0
assert infinite_predictions == 0
assert negative_predictions == 0

# ------------------------------------------------------------
# Prediction statistics
# ------------------------------------------------------------

prediction_series_full = pd.Series(
    predictions_full,
    name="Prediction"
)

print("\n" + "=" * 70)
print("COMPLETE PREDICTION STATISTICS")
print("=" * 70)

print(
    prediction_series_full.describe()
)

# ------------------------------------------------------------
# Check expected test IDs
# ------------------------------------------------------------

test_ids = test_complete["Id"].copy()

print("\nTest ID statistics:")
print(test_ids.describe())

print("\nUnique test IDs:")
print(test_ids.nunique())

assert len(test_ids) == 41088
assert test_ids.nunique() == 41088

# ------------------------------------------------------------
# Build prediction dataframe
# ------------------------------------------------------------

predictions_df = pd.DataFrame({
    "Id": test_ids.to_numpy(),
    "Store": test_complete["Store"].to_numpy(),
    "Date": test_complete["Date"].to_numpy(),
    "Open": test_complete["Open"].to_numpy(),
    "Promo": test_complete["Promo"].to_numpy(),
    "Prediction": predictions_full,
})

# ------------------------------------------------------------
# Validate prediction dataframe
# ------------------------------------------------------------

print("\nPrediction dataframe shape:")
print(predictions_df.shape)

assert predictions_df.shape == (41088, 6)

assert predictions_df["Id"].nunique() == 41088

assert (
    predictions_df["Prediction"]
    .isna()
    .sum()
    == 0
)

assert (
    np.isinf(
        predictions_df["Prediction"]
    ).sum()
    == 0
)

assert (
    (
        predictions_df["Prediction"] < 0
    ).sum()
    == 0
)

# ------------------------------------------------------------
# Check prediction ordering
# ------------------------------------------------------------

assert (
    predictions_df["Id"].tolist()
    == test_complete["Id"].tolist()
)

# ------------------------------------------------------------
# First 20 predictions
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FIRST 20 COMPLETE TEST PREDICTIONS")
print("=" * 70)

print(
    predictions_df[
        [
            "Id",
            "Store",
            "Date",
            "Open",
            "Promo",
            "Prediction",
        ]
    ].head(20).to_string(index=False)
)

# ------------------------------------------------------------
# Last 20 predictions
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LAST 20 COMPLETE TEST PREDICTIONS")
print("=" * 70)

print(
    predictions_df[
        [
            "Id",
            "Store",
            "Date",
            "Open",
            "Promo",
            "Prediction",
        ]
    ].tail(20).to_string(index=False)
)

# ------------------------------------------------------------
# Predictions by Open status
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PREDICTIONS BY OPEN STATUS")
print("=" * 70)

print(
    predictions_df
    .groupby("Open")["Prediction"]
    .agg(
        ["count", "mean", "std", "min", "max"]
    )
)

# ------------------------------------------------------------
# Verify all rows have predictions
# ------------------------------------------------------------

assert len(predictions_df) == 41088

assert (
    predictions_df["Prediction"].notna().all()
)

assert np.isfinite(
    predictions_df["Prediction"]
).all()

# ------------------------------------------------------------
# Final success
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("✓ STEP 23M.18 PASSED")
print("=" * 70)

print("Test rows:", len(predictions_df))
print("Predictions:", len(predictions_full))
print("Model features:", X_test_full.shape[1])

print("\n✓ 41,088 predictions generated")
print("✓ Row count matches test data")
print("✓ IDs are unique")
print("✓ No NaN predictions")
print("✓ No infinite predictions")
print("✓ No negative predictions")
print("✓ Prediction ordering preserved")
print("✓ Complete prediction dataframe ready")


STEP 23M.18 — GENERATE PREDICTIONS FOR ALL 41,088 TEST ROWS

Test feature matrix:
(41088, 26)

Model expected features:
26

✓ Input matrix validated
✓ Shape: (41088, 26)
✓ Feature names match
✓ Feature order matches
✓ No missing values

Generating predictions...

Prediction type:
<class 'numpy.ndarray'>

Prediction shape:
(41088,)

Prediction validation:
Missing predictions: 0
Infinite predictions: 0
Negative predictions: 0
Zero predictions: 5995

COMPLETE PREDICTION STATISTICS
count    41088.000000
mean      5986.127072
std       3598.542189
min          0.000000
25%       4177.752860
50%       6091.395072
75%       8000.347586
max      26572.582726
Name: Prediction, dtype: float64

Test ID statistics:
count    41088.000000
mean     20544.500000
std      11861.228267
min          1.000000
25%      10272.750000
50%      20544.500000
75%      30816.250000
max      41088.000000
Name: Id, dtype: float64

Unique test IDs:
41088

Prediction dataframe shape:
(41088, 6)

FIRST 20 COMPLETE TE

In [123]:
predictions = model.predict(X_test_full)

In [125]:
# ============================================================
# STEP 23M.18 — PERSIST COMPLETE TEST PREDICTIONS
# ============================================================

predictions_full = np.asarray(predictions).reshape(-1)

assert len(predictions_full) == 41088, (
    f"Expected 41088 predictions, got {len(predictions_full)}"
)

assert np.isfinite(predictions_full).all(), (
    "Complete test predictions contain NaN or infinite values."
)

assert (predictions_full >= 0).all(), (
    "Complete test predictions contain negative values."
)

prediction_df = test_complete[
    ["Id", "Store", "Date", "Open", "Promo"]
].copy()

prediction_df["Prediction"] = predictions_full

assert prediction_df.shape == (41088, 6)
assert prediction_df["Id"].nunique() == 41088

print("\n✓ Complete predictions persisted")
print(f"Prediction array: {predictions_full.shape}")
print(f"Prediction dataframe: {prediction_df.shape}")
print(prediction_df.head(10))


✓ Complete predictions persisted
Prediction array: (41088,)
Prediction dataframe: (41088, 6)
   Id  Store       Date  Open  Promo    Prediction
0   1      1 2015-09-17   1.0      1   4966.408767
1   2      3 2015-09-17   1.0      1   7753.229326
2   3      7 2015-09-17   1.0      1  12841.445303
3   4      8 2015-09-17   1.0      1   7423.199135
4   5      9 2015-09-17   1.0      1   7920.588151
5   6     10 2015-09-17   1.0      1   6532.019382
6   7     11 2015-09-17   1.0      1   9123.920922
7   8     12 2015-09-17   1.0      1   8347.599684
8   9     13 2015-09-17   1.0      1   7202.759809
9  10     14 2015-09-17   1.0      1   5978.026309


In [126]:
print("predictions_full:", len(predictions_full))
print("prediction_df:", prediction_df.shape)

assert len(predictions_full) == 41088
assert prediction_df.shape == (41088, 6)

print("\n✓ STEP 23M.18 output available for STEP 23M.19")

predictions_full: 41088
prediction_df: (41088, 6)

✓ STEP 23M.18 output available for STEP 23M.19


##### Step 23M.19 — Build and validate the final 41,088-row submission

In [127]:
# ============================================================
# STEP 23M.19 — BUILD AND VALIDATE FINAL SUBMISSION
# ============================================================

import numpy as np
import pandas as pd

print("=" * 70)
print("STEP 23M.19 — BUILD FINAL SUBMISSION")
print("=" * 70)

# ------------------------------------------------------------
# Validate persisted Step 23M.18 output
# ------------------------------------------------------------

assert "prediction_df" in globals(), (
    "prediction_df from STEP 23M.18 was not found."
)

assert "predictions_full" in globals(), (
    "predictions_full from STEP 23M.18 was not found."
)

assert len(predictions_full) == 41088, (
    f"Expected 41088 predictions, got {len(predictions_full)}"
)

assert prediction_df.shape == (41088, 6), (
    f"Unexpected prediction dataframe shape: "
    f"{prediction_df.shape}"
)

# ------------------------------------------------------------
# Build final submission
# ------------------------------------------------------------

submission = pd.DataFrame({
    "Id": prediction_df["Id"].astype(int).to_numpy(),
    "Sales": np.asarray(predictions_full, dtype=float)
})

print("\nSubmission shape:")
print(submission.shape)

print("\nSubmission columns:")
print(submission.columns.tolist())

# ------------------------------------------------------------
# Schema validation
# ------------------------------------------------------------

EXPECTED_COLUMNS = ["Id", "Sales"]

assert submission.shape == (41088, 2)

assert submission.columns.tolist() == EXPECTED_COLUMNS, (
    f"Unexpected submission columns: "
    f"{submission.columns.tolist()}"
)

# ------------------------------------------------------------
# ID validation
# ------------------------------------------------------------

assert submission["Id"].notna().all()
assert submission["Id"].is_unique
assert len(submission["Id"]) == 41088

assert submission["Id"].min() == 1
assert submission["Id"].max() == 41088

assert set(submission["Id"]) == set(range(1, 41089))

print("\nID validation:")
print(f"Rows: {len(submission):,}")
print(f"Unique IDs: {submission['Id'].nunique():,}")
print(f"Minimum ID: {submission['Id'].min()}")
print(f"Maximum ID: {submission['Id'].max()}")

# ------------------------------------------------------------
# Sales validation
# ------------------------------------------------------------

missing_sales = int(submission["Sales"].isna().sum())
infinite_sales = int(
    np.isinf(submission["Sales"].to_numpy()).sum()
)
negative_sales = int(
    (submission["Sales"] < 0).sum()
)

print("\nSales validation:")
print(f"Missing: {missing_sales}")
print(f"Infinite: {infinite_sales}")
print(f"Negative: {negative_sales}")

assert missing_sales == 0
assert infinite_sales == 0
assert negative_sales == 0

# ------------------------------------------------------------
# Prediction ordering validation
# ------------------------------------------------------------

assert np.array_equal(
    submission["Id"].to_numpy(),
    prediction_df["Id"].astype(int).to_numpy()
)

assert np.allclose(
    submission["Sales"].to_numpy(),
    predictions_full
)

print("\n✓ Prediction ordering preserved")
print("✓ Submission values match Step 23M.18 predictions")

# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------

print("\n======================================================================")
print("FINAL SUBMISSION STATISTICS")
print("======================================================================")

print(submission["Sales"].describe())

print("\nZero sales predictions:")
print(int((submission["Sales"] == 0).sum()))

print("\nFirst 10 rows:")
print(submission.head(10).to_string(index=False))

print("\nLast 10 rows:")
print(submission.tail(10).to_string(index=False))

# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

assert submission.shape == (41088, 2)
assert submission.columns.tolist() == ["Id", "Sales"]
assert submission["Id"].is_unique
assert submission["Id"].notna().all()
assert submission["Sales"].notna().all()
assert np.isfinite(submission["Sales"]).all()
assert (submission["Sales"] >= 0).all()

print("\n" + "=" * 70)
print("✓ STEP 23M.19 PASSED")
print("=" * 70)

print("Rows:       41,088")
print("Columns:    ['Id', 'Sales']")
print("Unique IDs: 41,088")
print("Missing:    0")
print("Infinite:   0")
print("Negative:   0")

print("\n✓ FINAL SUBMISSION DATAFRAME READY")

STEP 23M.19 — BUILD FINAL SUBMISSION

Submission shape:
(41088, 2)

Submission columns:
['Id', 'Sales']

ID validation:
Rows: 41,088
Unique IDs: 41,088
Minimum ID: 1
Maximum ID: 41088

Sales validation:
Missing: 0
Infinite: 0
Negative: 0

✓ Prediction ordering preserved
✓ Submission values match Step 23M.18 predictions

FINAL SUBMISSION STATISTICS
count    41088.000000
mean      5986.127072
std       3598.542189
min          0.000000
25%       4177.752860
50%       6091.395072
75%       8000.347586
max      26572.582726
Name: Sales, dtype: float64

Zero sales predictions:
5995

First 10 rows:
 Id        Sales
  1  4966.408767
  2  7753.229326
  3 12841.445303
  4  7423.199135
  5  7920.588151
  6  6532.019382
  7  9123.920922
  8  8347.599684
  9  7202.759809
 10  5978.026309

Last 10 rows:
   Id        Sales
41079  6291.327510
41080  4788.983900
41081  3391.266305
41082  6412.960825
41083  3950.422242
41084  2744.041770
41085  7463.406191
41086  5953.793072
41087 24772.135532
41088  7

##### STEP 23M.20 — SAVE AND VALIDATE FINAL SUBMISSION CSV

In [128]:
# ============================================================
# STEP 23M.20 — SAVE AND VALIDATE FINAL SUBMISSION CSV
# ============================================================

import os
import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 23M.20 — SAVE AND VALIDATE FINAL SUBMISSION CSV")
print("=" * 70)

# ------------------------------------------------------------
# Output path
# ------------------------------------------------------------

OUTPUT_PATH = "submission.csv"

# ------------------------------------------------------------
# Final dataframe validation before writing
# ------------------------------------------------------------

assert "submission" in globals(), (
    "Final submission dataframe was not found."
)

assert submission.shape == (41088, 2)
assert submission.columns.tolist() == ["Id", "Sales"]

assert submission["Id"].is_unique
assert submission["Id"].notna().all()
assert submission["Sales"].notna().all()

assert np.isfinite(submission["Sales"]).all()
assert (submission["Sales"] >= 0).all()

assert set(submission["Id"]) == set(range(1, 41089))

# ------------------------------------------------------------
# Save CSV
# ------------------------------------------------------------

submission.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\nCSV saved:")
print(os.path.abspath(OUTPUT_PATH))

# ------------------------------------------------------------
# File existence validation
# ------------------------------------------------------------

assert os.path.exists(OUTPUT_PATH), (
    "Submission CSV was not created."
)

file_size = os.path.getsize(OUTPUT_PATH)

print("\nFile validation:")
print(f"Exists: {os.path.exists(OUTPUT_PATH)}")
print(f"File size: {file_size:,} bytes")

assert file_size > 0

# ------------------------------------------------------------
# Reload CSV from disk
# ------------------------------------------------------------

submission_check = pd.read_csv(OUTPUT_PATH)

print("\nReloaded CSV shape:")
print(submission_check.shape)

print("\nReloaded CSV columns:")
print(submission_check.columns.tolist())

# ------------------------------------------------------------
# Validate actual CSV contents
# ------------------------------------------------------------

assert submission_check.shape == (41088, 2)

assert submission_check.columns.tolist() == [
    "Id",
    "Sales"
]

assert submission_check["Id"].is_unique
assert submission_check["Id"].notna().all()
assert submission_check["Sales"].notna().all()

assert np.isfinite(
    submission_check["Sales"].to_numpy()
).all()

assert (
    submission_check["Sales"] >= 0
).all()

assert submission_check["Id"].min() == 1
assert submission_check["Id"].max() == 41088

assert set(
    submission_check["Id"]
) == set(range(1, 41089))

# ------------------------------------------------------------
# Verify CSV values exactly match dataframe
# ------------------------------------------------------------

assert np.array_equal(
    submission_check["Id"].to_numpy(),
    submission["Id"].to_numpy()
)

assert np.allclose(
    submission_check["Sales"].to_numpy(),
    submission["Sales"].to_numpy()
)

# ------------------------------------------------------------
# Preview actual file contents
# ------------------------------------------------------------

print("\n======================================================================")
print("CSV PREVIEW")
print("======================================================================")

print("\nFirst 10 rows:")
print(
    submission_check.head(10).to_string(index=False)
)

print("\nLast 10 rows:")
print(
    submission_check.tail(10).to_string(index=False)
)

# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("✓ STEP 23M.20 PASSED")
print("=" * 70)

print(f"CSV file:       {os.path.abspath(OUTPUT_PATH)}")
print(f"File size:      {file_size:,} bytes")
print("Rows:           41,088")
print("Columns:        ['Id', 'Sales']")
print("Unique IDs:     41,088")
print("Missing values: 0")
print("Infinite:       0")
print("Negative:       0")

print("\n✓ CSV successfully written")
print("✓ CSV successfully reloaded")
print("✓ CSV contents match final submission dataframe")
print("✓ FINAL SUBMISSION FILE READY")

STEP 23M.20 — SAVE AND VALIDATE FINAL SUBMISSION CSV

CSV saved:
e:\Praxis\rossmann-sales\notebooks\submission.csv

File validation:
Exists: True
File size: 936,561 bytes

Reloaded CSV shape:
(41088, 2)

Reloaded CSV columns:
['Id', 'Sales']

CSV PREVIEW

First 10 rows:
 Id        Sales
  1  4966.408767
  2  7753.229326
  3 12841.445303
  4  7423.199135
  5  7920.588151
  6  6532.019382
  7  9123.920922
  8  8347.599684
  9  7202.759809
 10  5978.026309

Last 10 rows:
   Id        Sales
41079  6291.327510
41080  4788.983900
41081  3391.266305
41082  6412.960825
41083  3950.422242
41084  2744.041770
41085  7463.406191
41086  5953.793072
41087 24772.135532
41088  7946.851410

✓ STEP 23M.20 PASSED
CSV file:       e:\Praxis\rossmann-sales\notebooks\submission.csv
File size:      936,561 bytes
Rows:           41,088
Columns:        ['Id', 'Sales']
Unique IDs:     41,088
Missing values: 0
Infinite:       0
Negative:       0

✓ CSV successfully written
✓ CSV successfully reloaded
✓ CSV conten